# CS781 Phase 3 — Rhetorical Role prediction 

## 0. Write `rr.py`, `rr_hyb.py`, `ensemble3.py`, `evaluate_phase3.py`

In [1]:
%%writefile rr.py

"""
rr.py  --  CS781 Phase 3 (Rhetorical Role prediction) on top of YOUR Phase-1 RoPE+MQA+SwiGLU decoder.

One run = one adaptation method.  Examples:
  python rr.py --method lorma   --data_dir DATA --tokenizer bpe_tokenizer.json --init model_step45000.pt --out runs/lorma
  python rr.py --method full    ... --out runs/full
  python rr.py --method kd      ... --teacher runs/full/best.pt --out runs/kd

Task: every sentence of a legal judgment gets one of 13 rhetorical-role labels (metric = macro-F1 over all sentences).
Model: documents are cut into <=512-token windows of consecutive sentences (each sentence starts with a <s> marker);
       the Phase-1 trunk runs over the window, token states are mean-pooled per sentence, and a linear head sees
       [pooled sentence ++ relative position in the document ++ CL/IT flag].

  A. Phase-1 tokenizer + model code (copied from your notebooks; edits marked [P2-EDIT]/[P3-EDIT])
  B. Data, windows, metric
  C. Adaptation methods: full / lora / lorma(+) / adalora / ia3 / prompt / prefix / kd
  D. Training, inference, prediction writer
"""
import argparse, hashlib, json, math, os, pickle, random, re, sys, time, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

IGN = -100
PAD_ID = 256          
EOS_ID = 259          




class ByteLevelBPETokenizer:
    SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]

    def __init__(self):
        self.merges = {}
        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        self.special_to_id = {}
        for i, tok in enumerate(self.SPECIAL_TOKENS):
            tid = 256 + i
            self.special_to_id[tok] = tid
            self.id_to_bytes[tid] = tok.encode("utf-8")

    @property
    def vocab_size(self):
        return len(self.id_to_bytes)

    @staticmethod
    def _pair_counts(sequences):
        counts = {}
        for seq, freq in sequences:
            for pair in zip(seq, seq[1:]):
                counts[pair] = counts.get(pair, 0) + freq
        return counts

    @staticmethod
    def _apply_merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair:
                out.append(new_id); i += 2
            else:
                out.append(seq[i]); i += 1
        return out

    def train(self, text, vocab_size, max_words=100000, verbose=False):   
        num_merges = vocab_size - self.vocab_size
        word_freq = {}
        for word in text.split():
            word_freq[word] = word_freq.get(word, 0) + 1
        sequences = [[list(w.encode("utf-8")), f] for w, f in word_freq.items()]
        next_id = self.vocab_size
        for _ in range(num_merges):
            counts = self._pair_counts(sequences)
            if not counts: break
            best = max(counts, key=counts.get)
            self.merges[best] = next_id
            self.id_to_bytes[next_id] = self.id_to_bytes[best[0]] + self.id_to_bytes[best[1]]
            for e in sequences: e[0] = self._apply_merge(e[0], best, next_id)
            next_id += 1
        return self

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"merges": {f"{a},{b}": n for (a, b), n in self.merges.items()}}, f)

    @classmethod
    def load(cls, path):
        tok = cls()
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for pair_str, nid in data["merges"].items():
            a, b = (int(x) for x in pair_str.split(","))
            tok.merges[(a, b)] = nid
            tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return self.weight * x * torch.rsqrt(rms + self.eps)


def rotate_half(x):
    x1 = x[..., ::2]; x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, q, k):
        T = q.size(-2)
        cos = self.cos[:, :, :T, :]; sin = self.sin[:, :, :T, :]
        return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)


class MultiQueryCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % kv_heads == 0
        self.n_heads, self.kv_heads = n_heads, kv_heads
        self.d_head = d_model // n_heads
        self.kv_dim = kv_heads * self.d_head
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, self.kv_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.d_head, max_seq_len=max_seq_len, base=rope_base)
        self.dropout = dropout
        self.resid_dropout = nn.Dropout(dropout)
        self.causal = True                                   
        self.prefix_kv = None                                

    def forward(self, x, pad_mask=None):                     
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv_heads, self.d_head).transpose(1, 2)
        q, k = self.rope(q, k)
        P = 0
        if self.prefix_kv is not None:                       
            pk, pv = self.prefix_kv
            P = pk.size(0)
            pk = pk.permute(1, 0, 2).unsqueeze(0).expand(B, -1, -1, -1).to(k.dtype)
            pv = pv.permute(1, 0, 2).unsqueeze(0).expand(B, -1, -1, -1).to(v.dtype)
            k = torch.cat([pk, k], dim=2); v = torch.cat([pv, v], dim=2)
        if self.kv_heads != self.n_heads:
            r = self.n_heads // self.kv_heads
            k = k.repeat_interleave(r, dim=1); v = v.repeat_interleave(r, dim=1)
        dp = self.dropout if self.training else 0.0
        if self.causal and P == 0:                           
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=dp, is_causal=True)
        else:
            if self.causal:
                m = torch.ones(T, P + T, dtype=torch.bool, device=x.device)
                m[:, P:] = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
                m = m[None, None]
            elif pad_mask is None:
                m = None
            else:
                m = pad_mask if P == 0 else torch.cat([torch.ones(B, P, dtype=torch.bool, device=x.device), pad_mask], 1)
                m = m[:, None, None, :]                      
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=m, dropout_p=dp, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down(F.silu(self.gate(x)) * self.up(x)))


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base):
        super().__init__()
        self.norm_attn = RMSNorm(d_model)
        self.attn = MultiQueryCausalAttention(d_model, n_heads, kv_heads, max_seq_len, dropout, rope_base)
        self.norm_ffn = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, pad_mask=None):                     
        x = x + self.attn(self.norm_attn(x), pad_mask)
        x = x + self.ffn(self.norm_ffn(x))
        return x


class RoPEMQADecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, kv_heads, d_ff, max_seq_len,
                 dropout=0.1, rope_base=10000.0, tie_weights=True):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base)
                                     for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_weights:
            self.lm_head.weight = self.token_emb.weight


DEFAULT_LM_CFG = dict(vocab_size=12000, d_model=768, n_layers=12, n_heads=12, kv_heads=1,
                      d_ff=3072, max_seq_len=512, dropout=0.1, rope_base=10000.0)


def build_lm(cfg):
    c = {**DEFAULT_LM_CFG, **{k: v for k, v in cfg.items() if k in DEFAULT_LM_CFG}}
    return RoPEMQADecoderLM(**c, tie_weights=True), c


def load_phase1(path, n_layers=None, layer_map=None):
    """Load your Phase-1 weights.  Accepts model.pt ({state_dict, config}), checkpoint.pt ({model_state, run_config})
    or a raw state_dict (model_stepXXXXX.pt).  Optionally keeps only `layer_map` blocks (for the KD student)."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = {}
    if isinstance(ck, dict) and "state_dict" in ck:
        sd, cfg = ck["state_dict"], ck.get("config", {})
    elif isinstance(ck, dict) and "model_state" in ck:
        sd = ck["model_state"]
        rc = ck.get("run_config", {})
        cfg = dict(vocab_size=rc.get("VOCAB_SIZE"), d_model=rc.get("D_MODEL"), n_layers=rc.get("N_LAYERS"),
                   n_heads=rc.get("N_HEADS"), d_ff=rc.get("D_FF"), max_seq_len=rc.get("BLOCK_SIZE"),
                   kv_heads=rc.get("KV_HEADS"), rope_base=rc.get("ROPE_BASE"))
        cfg = {k: v for k, v in cfg.items() if v is not None}
    else:
        sd = ck
    sd = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in sd.items()}
    cfg["n_layers"] = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))
    cfg["vocab_size"], cfg["d_model"] = sd["token_emb.weight"].shape
    cfg["d_ff"] = sd["blocks.0.ffn.gate.weight"].shape[0]
    cfg["kv_heads"] = sd["blocks.0.attn.k_proj.weight"].shape[0] // (cfg["d_model"] // cfg.get("n_heads", 12))
    if layer_map is not None:                                
        new = {k: v for k, v in sd.items() if not k.startswith("blocks.")}
        for ni, oi in enumerate(layer_map):
            for k, v in sd.items():
                if k.startswith(f"blocks.{oi}."):
                    new[f"blocks.{ni}." + k.split(".", 2)[2]] = v
        sd, cfg["n_layers"] = new, len(layer_map)
    lm, cfg = build_lm(cfg)
    missing, unexpected = lm.load_state_dict(sd, strict=False)
    bad = [k for k in missing if k != "lm_head.weight"] + list(unexpected)
    assert not bad, f"checkpoint/architecture mismatch: {bad[:6]}"
    return lm, cfg







BOS_ID = 258          
POS_BINS = 20
PAD_MULTIPLE = 1      
PAD_CAP = 1 << 30     


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_vocab(data):
    """Same acceptance rules as evaluate_phase3.py: list | label->id | id->label.  Returns (labels, label2idx)."""
    if isinstance(data, list):
        labels = list(data)
    elif isinstance(data, dict) and data:
        vals = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in vals):
            labels = [k for k, _ in sorted(data.items(), key=lambda kv: kv[1])]
        elif all(isinstance(v, str) for v in vals):
            labels = [v for _, v in sorted(data.items(), key=lambda kv: int(kv[0]) if str(kv[0]).isdigit() else kv[0])]
        else:
            raise ValueError("unsupported label_vocab.json format")
    else:
        raise ValueError("label_vocab.json must be a non-empty list or object")
    assert len(labels) == 13 and len(set(labels)) == 13, f"expected 13 unique labels, got {labels}"
    return labels, {l: i for i, l in enumerate(labels)}


def iter_raw_docs(obj):
    """Accepts  [ {id,text,labels}, ... ]   or   { id: {text,labels} }   or   { id: [sentences] }  -> (id, sentences, labels|None)."""
    def pick(d, names):
        for n in names:
            if n in d: return d[n]
        return None
    if isinstance(obj, dict) and "documents" in obj and isinstance(obj["documents"], (list, dict)):
        obj = obj["documents"]
    if isinstance(obj, list):
        for k, d in enumerate(obj):
            did = pick(d, ("id", "doc_id", "document_id", "name", "file"))
            did = str(did) if did is not None else str(k)
            yield did, pick(d, ("text", "sentences", "sents", "sentence")), pick(d, ("labels", "label", "roles", "rhetorical_roles", "annotations"))
    elif isinstance(obj, dict):
        for did, d in obj.items():
            if isinstance(d, dict):
                yield str(did), pick(d, ("text", "sentences", "sents", "sentence")), pick(d, ("labels", "label", "roles", "rhetorical_roles", "annotations"))
            elif isinstance(d, list):
                yield str(did), d, None
            else:
                raise ValueError(f"cannot parse document {did!r}: {type(d)}")
    else:
        raise ValueError(f"cannot parse top-level {type(obj)}")


def label_to_idx(v, label2idx, labels):
    if isinstance(v, bool): raise ValueError(v)
    if isinstance(v, int): return v
    if isinstance(v, str):
        if v in label2idx: return label2idx[v]
        if v.isdigit(): return int(v)
    raise ValueError(f"unknown label {v!r}")


class WordBPE:
    """Phase-1 encode(): whitespace words -> byte-BPE (min-rank merge loop) -> </s> after every word.  Words are cached."""
    def __init__(self, tok):
        self.tok, self.cache = tok, {}

    def word(self, w):
        r = self.cache.get(w)
        if r is None:
            tok = self.tok
            seq = list(w.encode("utf-8"))
            while len(seq) > 1:
                mergeable = [p for p in zip(seq, seq[1:]) if p in tok.merges]
                if not mergeable: break
                pair = min(mergeable, key=lambda p: tok.merges[p])
                seq = tok._apply_merge(seq, pair, tok.merges[pair])
            r = tuple(seq) + (EOS_ID,)
            self.cache[w] = r
        return r

    def sentence(self, s):
        out = []
        for w in str(s).split(): out.extend(self.word(w))
        return np.array(out if out else [EOS_ID], np.int32)


def build_docs(raw, dom, label2idx, labels, enc, tag=""):
    docs, bad = [], 0
    for did, sents, labs in raw:
        if not isinstance(sents, list): raise ValueError(f"{tag} doc {did}: sentences field is {type(sents)}")
        toks = [enc.sentence(s) for s in sents]
        y = None
        if labs is not None:
            if len(labs) != len(sents): bad += 1; labs = (list(labs) + ["None"] * len(sents))[:len(sents)]
            y = np.array([label_to_idx(l, label2idx, labels) for l in labs], np.int64)
        docs.append(dict(id=did, dom=dom, raw=toks, y=y, n=len(sents)))
    if bad: print(f"  WARNING {tag}: {bad} docs had len(labels) != len(sentences) (padded/truncated)")
    return docs


def finalize_docs(docs, max_sent, marker):
    """Truncate long sentences (keep head 2/3 + tail 1/3), add the sentence-start marker; fills d['sents'], d['lens']."""
    for d in docs:
        ss = []
        for t in d["raw"]:
            if len(t) > max_sent:
                h = (max_sent * 2) // 3
                t = np.concatenate([t[:h], t[len(t) - (max_sent - h):]])
            if marker: t = np.concatenate([np.array([BOS_ID], np.int32), t])
            ss.append(t)
        d["sents"], d["lens"] = ss, [len(t) for t in ss]
    return docs


def make_windows(lens, budget, core_frac=0.5, first_frac=None):
    """Partition the sentences into CORE chunks (<= core_frac*budget tokens) and extend each with left/right context sentences
    until `budget` tokens are used.  Returns [(lo, hi, a, b)] : window = sentences lo..hi, loss/prediction on core a..b."""
    n = len(lens); cb = max(1, int(budget * core_frac)); wins = []; i = 0
    cur = cb if first_frac is None else max(1, int(cb * first_frac))
    while i < n:
        j, tot = i, 0
        while j < n and (j == i or tot + lens[j] <= cur):
            tot += lens[j]; j += 1
        lo, hi, used = i, j, tot
        grow = True
        while grow:
            grow = False
            if lo > 0 and used + lens[lo - 1] <= budget: lo -= 1; used += lens[lo]; grow = True
            if hi < n and used + lens[hi] <= budget: used += lens[hi]; hi += 1; grow = True
        wins.append((lo, hi, i, j)); i = j; cur = cb
    return wins


def collate(items, docs, device, train_labels=True):
    """items: [(doc_idx, lo, hi, a, b)] -> tensors.  Labels only on the core a..b (IGN elsewhere)."""
    B = len(items)
    toks = [np.concatenate(docs[di]["sents"][lo:hi]) for di, lo, hi, a, b in items]
    L = max(len(t) for t in toks); S = max(hi - lo for _, lo, hi, _, _ in items)
    if PAD_MULTIPLE > 1: L = min(-(-L // PAD_MULTIPLE) * PAD_MULTIPLE, PAD_CAP)   
    ids = torch.full((B, L), PAD_ID, dtype=torch.long)
    sid = torch.full((B, L), -1, dtype=torch.long)
    msk = torch.zeros((B, L), dtype=torch.bool)
    lab = torch.full((B, S), IGN, dtype=torch.long)
    pb = torch.zeros((B, S), dtype=torch.long)
    dom = torch.zeros((B,), dtype=torch.long)
    for k, ((di, lo, hi, a, b), t) in enumerate(zip(items, toks)):
        d = docs[di]
        ids[k, :len(t)] = torch.from_numpy(t.astype(np.int64)); msk[k, :len(t)] = True
        pos = 0
        for s in range(lo, hi):
            ln = d["lens"][s]; sid[k, pos:pos + ln] = s - lo; pos += ln
        idx = np.arange(lo, hi)
        pb[k, :hi - lo] = torch.from_numpy(np.minimum((idx * POS_BINS) // max(1, d["n"]), POS_BINS - 1).astype(np.int64))
        if train_labels and d["y"] is not None: lab[k, a - lo:b - lo] = torch.from_numpy(d["y"][a:b])
        dom[k] = d["dom"]
    return ids.to(device), sid.to(device), (None if msk.all() else msk.to(device)), pb.to(device), dom.to(device), lab.to(device)


def macro_prf(y_true, y_pred, K):
    """Identical to sklearn f1/precision/recall(labels=all K, average='macro', zero_division=0)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    cm = np.bincount(y_true * K + y_pred, minlength=K * K).reshape(K, K)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(1, 2 * tp + fp + fn), 0.0)
    p = np.where(tp + fp > 0, tp / np.maximum(1, tp + fp), 0.0)
    r = np.where(tp + fn > 0, tp / np.maximum(1, tp + fn), 0.0)
    return dict(F1=float(f1.mean()), P=float(p.mean()), R=float(r.mean()), per_f1=f1.tolist(), per_p=p.tolist(), per_r=r.tolist(),
                support=cm.sum(1).tolist())


def score_docs(docs, lps):
    y = np.concatenate([d["y"] for d in docs]); p = np.concatenate([lp.argmax(1) for lp in lps])
    return macro_prf(y, p, lps[0].shape[1])





class RRModel(nn.Module):
    """Phase-1 decoder trunk + sentence pooling + (pooled ++ position-in-document ++ CL/IT) -> 13-way linear head."""
    def __init__(self, lm, K, causal=False, pool="mean", head_dropout=0.1, pos_dim=32, dom_dim=8):
        super().__init__()
        self.token_emb, self.blocks, self.final_norm = lm.token_emb, lm.blocks, lm.final_norm
        D = self.token_emb.embedding_dim
        self.pool = pool
        self.pos_emb = nn.Embedding(POS_BINS, pos_dim); self.dom_emb = nn.Embedding(2, dom_dim)
        nn.init.normal_(self.pos_emb.weight, std=0.02); nn.init.normal_(self.dom_emb.weight, std=0.02)
        self.drop = nn.Dropout(head_dropout)
        self.head = nn.Linear(D + pos_dim + dom_dim, K)
        self.prompt = None          
        self.prefix = None          
        self.set_causal(causal)

    def set_causal(self, causal):
        self.causal = causal
        for b in self.blocks: b.attn.causal = causal

    def sentence_reps(self, h, sid, S):
        B, L, D = h.shape
        valid = sid >= 0
        flat = (torch.arange(B, device=h.device)[:, None] * S + sid.clamp(min=0))[valid]
        hv = h[valid].float()
        if self.pool == "mean":
            sums = torch.zeros(B * S, D, device=h.device).index_add_(0, flat, hv)
            cnt = torch.zeros(B * S, device=h.device).index_add_(0, flat, torch.ones_like(flat, dtype=torch.float))
            return (sums / cnt.clamp(min=1)[:, None]).view(B, S, D)
        pos = torch.arange(L, device=h.device)[None].expand(B, L)[valid]
        if self.pool == "last":
            sel = torch.full((B * S,), -1, device=h.device, dtype=torch.long).scatter_reduce_(0, flat, pos, "amax", include_self=True)
        else:   
            sel = torch.full((B * S,), L, device=h.device, dtype=torch.long).scatter_reduce_(0, flat, pos, "amin", include_self=True)
        sel = sel.clamp(0, L - 1)
        base = (torch.arange(B, device=h.device)[:, None] * L).expand(B, S).reshape(-1)
        return h.float().reshape(B * L, D)[base + sel].view(B, S, D)

    def trunk(self, ids, pad_mask):
        """token ids -> final-normed hidden states (B, L, D) for the REAL tokens (prompt positions stripped)."""
        B = ids.size(0)
        x = self.token_emb(ids); P = 0
        if self.prompt is not None:
            P = self.prompt.size(0)
            x = torch.cat([self.prompt.unsqueeze(0).expand(B, -1, -1).to(x.dtype), x], 1)
            if pad_mask is not None: pad_mask = torch.cat([torch.ones(B, P, dtype=torch.bool, device=ids.device), pad_mask], 1)
        if self.prefix is not None:
            kv = self.prefix()
            for b, (pk, pv) in zip(self.blocks, kv): b.attn.prefix_kv = (pk, pv)
        for b in self.blocks: x = b(x, pad_mask)
        return self.final_norm(x)[:, P:]

    def forward(self, ids, sid, pad_mask, pos_bin, dom):
        B = ids.size(0); S = pos_bin.size(1)
        h = self.trunk(ids, pad_mask)
        rep = self.sentence_reps(h, sid, S)
        feat = torch.cat([rep, self.pos_emb(pos_bin), self.dom_emb(dom)[:, None, :].expand(B, S, -1)], -1)
        return self.head(self.drop(feat))


class PrefixEncoder(nn.Module):
    """Prefix tuning (Li & Liang 2021): P virtual key/value vectors per layer (kv_heads x d_head), optionally produced by a
    reparametrisation MLP (hidden>0) which is more stable to train than free parameters."""
    def __init__(self, n_layers, n_prefix, kv_heads, d_head, d_model, hidden):
        super().__init__()
        self.L, self.P, self.kvh, self.dh = n_layers, n_prefix, kv_heads, d_head
        out = n_layers * 2 * kv_heads * d_head
        if hidden > 0:
            self.emb = nn.Parameter(torch.randn(n_prefix, d_model) * 0.02)
            self.mlp = nn.Sequential(nn.Linear(d_model, hidden), nn.Tanh(), nn.Linear(hidden, out))
            self.direct = None
        else:
            self.direct = nn.Parameter(torch.randn(n_prefix, out) * 0.02)

    def forward(self):
        z = self.direct if self.direct is not None else self.mlp(self.emb)
        z = z.view(self.P, self.L, 2, self.kvh, self.dh)
        return [(z[:, l, 0], z[:, l, 1]) for l in range(self.L)]


class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) B A x   (Hu et al. 2021)"""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A), self.B)


class LoRMAPlusLinear(nn.Module):
    """LoRMA_+ (Bihany et al. 2025): W' = ((alpha/r) B A + I) W0 -- multiplicative, output side.  h = W0 x ; y = h + s B (A h)."""
    def __init__(self, base, r, alpha):
        super().__init__()
        d = base.out_features
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))

    def forward(self, x):
        h = self.base(x)
        return h + self.s * F.linear(F.linear(h, self.A), self.B)


class AdaLoRALinear(nn.Module):
    """AdaLoRA (Zhang et al. 2023): dW = P diag(E) Q with a rank-r SVD-like triplet structure; triplets are pruned by importance.
    A = Q (r x in), B = P (out x r), E = singular values (r).  `mask` (r) is the pruning state set by AdaLoRACtl."""
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base, self.r, self.s = base, r, alpha / r
        self.E = nn.Parameter(torch.zeros(r))
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.02)
        self.B = nn.Parameter(torch.randn(base.out_features, r) * 0.02)
        self.register_buffer("mask", torch.ones(r))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        z = F.linear(self.drop(x), self.A) * (self.E * self.mask)
        return self.base(x) + self.s * F.linear(z, self.B)


class AdaLoRACtl:
    """Importance-based rank allocation: sensitivity |w * grad| smoothed with EMA (beta1) times its uncertainty (beta2);
    triplet score = s(E_i) + mean_j s(A_ij) + mean_j s(B_ji).  Global budget follows a cubic schedule from init_r*M to target_r*M."""
    def __init__(self, model, init_r, target_r, tinit, tfinal, delta_t=10, beta1=0.85, beta2=0.85, orth=0.1):
        self.mods = [m for m in model.modules() if isinstance(m, AdaLoRALinear)]
        self.n_total = sum(m.r for m in self.mods)
        self.init_budget, self.target_budget = self.n_total, int(round(target_r / init_r * self.n_total))
        self.tinit, self.tfinal, self.dt, self.b1, self.b2, self.orth = tinit, tfinal, delta_t, beta1, beta2, orth
        self.ipt, self.unc = {}, {}
        self.budget = self.init_budget

    def orth_loss(self):
        if not self.mods or self.orth <= 0: return 0.0
        tot = 0.0
        for m in self.mods:
            A, B = m.A.float(), m.B.float(); I = torch.eye(m.r, device=A.device)
            tot = tot + torch.linalg.matrix_norm(A @ A.T - I) + torch.linalg.matrix_norm(B.T @ B - I)
        return self.orth * tot / len(self.mods)

    def budget_at(self, step, total):
        if step <= self.tinit: return self.init_budget
        if step > total - self.tfinal: return self.target_budget
        c = 1 - (step - self.tinit) / max(1, total - self.tfinal - self.tinit)
        return int((self.init_budget - self.target_budget) * c ** 3 + self.target_budget)

    @torch.no_grad()
    def step(self, step, total):
        """Call after backward (gradients already unscaled), before the optimiser step."""
        final_start = total - self.tfinal
        if step > final_start + 1: return                                              
        for m in self.mods:
            for p in (m.E, m.A, m.B):
                if p.grad is None or not bool(torch.isfinite(p.grad).all()): return    
        for m in self.mods:
            for p in (m.E, m.A, m.B):
                k = id(p); ipt = (p * p.grad).abs()
                a = self.ipt.get(k); a = ipt.clone() if a is None else self.b1 * a + (1 - self.b1) * ipt
                u = self.unc.get(k); u = torch.zeros_like(ipt) if u is None else u
                u = self.b2 * u + (1 - self.b2) * (ipt - a).abs()
                self.ipt[k], self.unc[k] = a, u
        self.budget = self.budget_at(step, total)
        if not (step == final_start + 1 or (step > self.tinit and step % self.dt == 0 and step <= final_start)): return
        scores = []
        for m in self.mods:
            sc = lambda p: self.ipt[id(p)] * self.unc[id(p)]
            scores.append(sc(m.E) + sc(m.A).mean(1) + sc(m.B).mean(0))
        allv = torch.cat(scores)
        k = self.n_total - self.budget
        if k <= 0:
            for m in self.mods: m.mask.fill_(1.0)
            return
        thr = torch.kthvalue(allv, k).values
        for m, sc_ in zip(self.mods, scores): m.mask.copy_((sc_ > thr).float())

    def rank_report(self):
        act = [int(m.mask.sum().item()) for m in self.mods]
        return dict(total_active=sum(act), total_possible=self.n_total, mean_rank=sum(act) / max(1, len(act)))


class IA3Out(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x): return self.base(x) * self.l


class IA3In(nn.Module):
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x): return self.base(x * self.l)


ATTN_T = ("q_proj", "k_proj", "v_proj", "out_proj")
FFN_T = ("gate", "up", "down")
HEAD_PREFIXES = ("head.", "pos_emb.", "dom_emb.")


def is_head_param(name): return name.startswith(HEAD_PREFIXES)


def apply_method(model, args, total_steps_hint=1000):
    """Freezes the trunk and installs the chosen adaptation.  Returns an optional AdaLoRACtl."""
    m = args.method
    for p in model.parameters(): p.requires_grad = False
    for n, p in model.named_parameters():
        if is_head_param(n): p.requires_grad = True                                
    tg = [t.strip() for t in args.targets.split(",") if t.strip()]

    def wrap_linears(factory, names):
        for b in model.blocks:
            for t in names:
                parent = b.attn if t in ATTN_T else b.ffn
                setattr(parent, t, factory(getattr(parent, t)))

    ctl = None
    if m in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    elif m == "lora":
        wrap_linears(lambda l: LoRALinear(l, args.rank, args.alpha, args.lora_dropout), tg)
    elif m == "lorma":
        wrap_linears(lambda l: LoRMAPlusLinear(l, args.rank, args.alpha), tg)
    elif m == "adalora":
        wrap_linears(lambda l: AdaLoRALinear(l, args.ada_init_r, args.alpha_ada, args.lora_dropout), tg)
        ctl = "pending"                                                           
    elif m == "ia3":
        for b in model.blocks:
            for t in [x.strip() for x in args.ia3_targets.split(",") if x.strip()]:
                parent = b.attn if t in ATTN_T else b.ffn
                lin = getattr(parent, t)
                setattr(parent, t, IA3In(lin) if t == "down" else IA3Out(lin))   
    elif m == "prompt":
        ids = torch.randint(260, model.token_emb.num_embeddings, (args.n_prompt,))
        model.prompt = nn.Parameter(model.token_emb.weight[ids].detach().clone())   
    elif m == "prefix":
        a0 = model.blocks[0].attn
        model.prefix = PrefixEncoder(len(model.blocks), args.n_prefix, a0.kv_heads, a0.d_head, model.token_emb.embedding_dim, args.prefix_hidden)
    else:
        raise ValueError(m)
    return ctl


def mark_new_params_trainable(model, before_ids):
    for p in model.parameters():
        if id(p) not in before_ids: p.requires_grad = True


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr





SPLITS = [("CL", 0), ("IT", 1)]


def load_split(data_dir, kind, enc, labels, label2idx):
    """kind in train/dev/test -> list of docs (CL first, then IT), in file order."""
    docs = []
    for dom_name, dom in SPLITS:
        p = os.path.join(data_dir, f"{dom_name}_{kind}.json")
        docs += build_docs(list(iter_raw_docs(load_json(p))), dom, label2idx, labels, enc, f"{dom_name}_{kind}")
    ids = [d["id"] for d in docs]
    assert len(set(ids)) == len(ids), f"duplicate document ids across CL/IT {kind}: {[i for i in ids if ids.count(i) > 1][:5]}"
    return docs


def get_docs(args, enc, labels, label2idx, tok_path):
    key = hashlib.md5(open(tok_path, "rb").read()[:100000] + b"rr-v1").hexdigest()[:10]
    cache = args.cache or os.path.join(args.data_dir, f"rrcache_{key}.pkl")
    if os.path.exists(cache):
        with open(cache, "rb") as f: out = pickle.load(f)
        print(f"  loaded tokenised cache {cache}")
        return out
    out = {k: load_split(args.data_dir, k, enc, labels, label2idx) for k in ("train", "dev", "test")}
    tmp = f"{cache}.{os.getpid()}.tmp"
    with open(tmp, "wb") as f: pickle.dump(out, f)
    os.replace(tmp, cache)
    return out


def batch_forward(model, docs, items, device, train_labels=True):
    ids, sid, msk, pb, dom, lab = collate(items, docs, device, train_labels)
    return model(ids, sid, msk, pb, dom), lab


@torch.no_grad()
def infer_docs(model, docs, budget, core_frac, device, amp=True, bs=16):
    """Per-document log-probs (n_sent, K).  Every sentence is the CORE of exactly one window and gets context from its neighbours."""
    was = model.training; model.eval()
    items = []
    for di, d in enumerate(docs):
        for lo, hi, a, b in make_windows(d["lens"], budget, core_frac):
            items.append((di, lo, hi, a, b, int(sum(d["lens"][lo:hi]))))
    items.sort(key=lambda t: t[-1])
    K = model.head.out_features
    out = [np.zeros((d["n"], K), np.float32) for d in docs]
    for i in range(0, len(items), bs):
        chunk = items[i:i + bs]
        with torch.autocast("cuda", enabled=amp):
            logits, _ = batch_forward(model, docs, [c[:5] for c in chunk], device, train_labels=False)
        lp = F.log_softmax(logits.float(), -1).cpu().numpy()
        for k, (di, lo, hi, a, b, _) in enumerate(chunk): out[di][a:b] = lp[k, a - lo:b - lo]
    if was: model.train()
    return out


def lr_at(step, total, warm, base):
    if step < warm: return base * (step + 1) / max(1, warm)
    p = (step - warm) / max(1, total - warm)
    return base * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))


def meta_of(args, lm_cfg, labels, model):
    return dict(method=args.method, causal=model.causal, pool=model.pool, marker=args.marker, max_sent=args.max_sent,
                lm_cfg=lm_cfg, labels=labels, n_layers=len(model.blocks))


def load_teacher(path, device):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    meta = ck["meta"]; assert meta["method"] == "full", "KD teacher must be a --method full run (best.pt)"
    lm, _ = build_lm(meta["lm_cfg"])
    model = RRModel(lm, len(meta["labels"]), causal=meta["causal"], pool=meta["pool"])
    model.load_state_dict(ck["state"]); return model.to(device).eval(), meta


def write_prediction(pred, docs, labels, outdir, name="prediction.json"):
    """pred: {doc_id: [label names]} ; validates ids / lengths / labels then writes prediction.json + submission.zip."""
    validate_prediction(pred, docs, labels)
    pj = os.path.join(outdir, name)
    with open(pj, "w", encoding="utf-8") as f: json.dump(pred, f)
    with zipfile.ZipFile(os.path.join(outdir, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
    return pj


def validate_prediction(pred, docs, labels):
    ok = set(labels)
    assert set(pred) == {d["id"] for d in docs}, "prediction ids != document ids"
    for d in docs:
        p = pred[d["id"]]
        assert isinstance(p, list) and len(p) == d["n"], f"{d['id']}: {len(p)} labels for {d['n']} sentences"
        assert all(x in ok for x in p), f"{d['id']}: unknown label"


def decode(lps, docs, labels):
    return {d["id"]: [labels[i] for i in lp.argmax(1)] for d, lp in zip(docs, lps)}


def save_lp(path_prefix, docs, lps):
    np.savez_compressed(path_prefix + "_lp.npz", lp=np.concatenate(lps, 0), lens=np.array([len(x) for x in lps]))
    with open(path_prefix + "_ids.json", "w") as f: json.dump([d["id"] for d in docs], f)


def trainable_state(model):
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in keep or k.endswith(".mask")}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True); ap.add_argument("--tokenizer", required=True)
    ap.add_argument("--init", required=True, help="Phase-1 weights (model_step45000.pt / model.pt / checkpoint.pt)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--method", required=True, choices=["full", "lora", "lorma", "adalora", "ia3", "prompt", "prefix", "kd"])
    ap.add_argument("--attn", default="bidir", choices=["causal", "bidir"])
    ap.add_argument("--pool", default="auto", choices=["auto", "mean", "last", "first"], help="auto: mean for bidir, last for causal")
    ap.add_argument("--epochs", type=float, default=6.0); ap.add_argument("--bs", type=int, default=8)
    ap.add_argument("--lr", type=float, default=None); ap.add_argument("--lr_head", type=float, default=1e-3)
    ap.add_argument("--wd", type=float, default=0.01); ap.add_argument("--warmup", type=int, default=100)
    ap.add_argument("--window", type=int, default=512, help="tokens per window incl. prompt tokens (<= Phase-1 context)")
    ap.add_argument("--max_sent", type=int, default=128, help="max tokens per sentence (head 2/3 + tail 1/3 kept)")
    ap.add_argument("--marker", type=int, default=1, help="prepend <s> as a sentence-start marker")
    ap.add_argument("--train_core_frac", type=float, default=1.0); ap.add_argument("--infer_core_frac", type=float, default=0.5)
    ap.add_argument("--cw_pow", type=float, default=0.5, help="class weight = (1/freq)^cw_pow, 0 = unweighted")
    ap.add_argument("--cw_max", type=float, default=10.0)
    ap.add_argument("--rank", type=int, default=16); ap.add_argument("--alpha", type=float, default=32)
    ap.add_argument("--lora_dropout", type=float, default=0.05)
    ap.add_argument("--targets", default="q_proj,k_proj,v_proj,out_proj,gate,up,down")
    ap.add_argument("--ada_init_r", type=int, default=24); ap.add_argument("--ada_target_r", type=int, default=16)
    ap.add_argument("--alpha_ada", type=float, default=48); ap.add_argument("--ada_orth", type=float, default=0.1)
    ap.add_argument("--ada_tinit_frac", type=float, default=0.1); ap.add_argument("--ada_tfinal_frac", type=float, default=0.3)
    ap.add_argument("--ia3_targets", default="k_proj,v_proj,down")
    ap.add_argument("--n_prompt", type=int, default=32); ap.add_argument("--n_prefix", type=int, default=32)
    ap.add_argument("--prefix_hidden", type=int, default=512, help="prefix reparametrisation MLP width (0 = free parameters)")
    ap.add_argument("--teacher", default=None); ap.add_argument("--student_layers", type=int, default=6)
    ap.add_argument("--kd_T", type=float, default=2.0); ap.add_argument("--kd_alpha", type=float, default=0.5)
    ap.add_argument("--full_data", action="store_true", help="also train on the dev docs; no model selection (use last epoch)")
    ap.add_argument("--time_budget_min", type=float, default=60.0)
    ap.add_argument("--eval_per_epoch", type=int, default=1)
    ap.add_argument("--max_train_docs", type=int, default=0); ap.add_argument("--max_dev_docs", type=int, default=0)
    ap.add_argument("--no_test", action="store_true")
    ap.add_argument("--seed", type=int, default=42); ap.add_argument("--cache", default=None)
    args = ap.parse_args()

    t_global = time.time()
    os.makedirs(args.out, exist_ok=True)
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    default_lr = dict(full=5e-5, kd=1e-4, lora=1e-3, lorma=1e-3, adalora=1e-3, ia3=1e-2, prompt=2e-2, prefix=5e-3)
    base_lr = args.lr if args.lr is not None else default_lr[args.method]
    print(f"== method={args.method} attn={args.attn} lr={base_lr} lr_head={args.lr_head} device={device}")

    
    tok = ByteLevelBPETokenizer.load(args.tokenizer)
    labels, label2idx = load_vocab(load_json(os.path.join(args.data_dir, "label_vocab.json")))
    K = len(labels)
    enc = WordBPE(tok)
    D_ = get_docs(args, enc, labels, label2idx, args.tokenizer)
    train_docs, dev_docs, test_docs = D_["train"], D_["dev"], D_["test"]
    if args.max_train_docs: train_docs = train_docs[:args.max_train_docs // 2] + [d for d in train_docs if d["dom"] == 1][:args.max_train_docs - args.max_train_docs // 2]
    if args.max_dev_docs: dev_docs = dev_docs[:args.max_dev_docs // 2] + [d for d in dev_docs if d["dom"] == 1][:args.max_dev_docs - args.max_dev_docs // 2]
    if args.full_data: train_docs = train_docs + dev_docs

    
    teacher = None
    if args.method == "kd":
        assert args.teacher, "--teacher runs/<full_run>/best.pt is required for kd"
        teacher, tmeta = load_teacher(args.teacher, device)
        layer_map = [int(round(x)) for x in np.linspace(0, tmeta["lm_cfg"]["n_layers"] - 1, args.student_layers)]
        print(f"  KD student = teacher blocks {layer_map}; teacher params={count_params(teacher)[0] / 1e6:.1f}M")
        lm, lm_cfg = load_phase1(args.init, layer_map=layer_map)
        args.attn = "causal" if tmeta["causal"] else "bidir"      
        args.pool, args.marker, args.max_sent = tmeta["pool"], tmeta["marker"], tmeta["max_sent"]
    else:
        lm, lm_cfg = load_phase1(args.init)
    pool = args.pool if args.pool != "auto" else ("last" if args.attn == "causal" else "mean")
    n_virtual = args.n_prompt if args.method == "prompt" else 0
    budget = args.window - n_virtual
    globals()["PAD_CAP"] = budget
    assert args.window <= lm_cfg["max_seq_len"], "window longer than the Phase-1 context (RoPE tables are built for 512)"
    assert args.max_sent + 1 <= budget, "max_sent must fit into one window"
    finalize_docs(train_docs, args.max_sent, args.marker); finalize_docs(dev_docs, args.max_sent, args.marker); finalize_docs(test_docs, args.max_sent, args.marker)
    cnt = np.zeros(K)
    for d in train_docs: cnt += np.bincount(d["y"], minlength=K)
    print(f"  docs: train={len(train_docs)} (CL={sum(d['dom'] == 0 for d in train_docs)}, IT={sum(d['dom'] == 1 for d in train_docs)}) dev={len(dev_docs)} test={len(test_docs)}"
          f" | sentences train={int(cnt.sum())} dev={sum(d['n'] for d in dev_docs)} test={sum(d['n'] for d in test_docs)}")
    print("  train label counts:", {labels[i]: int(c) for i, c in enumerate(cnt)})

    model = RRModel(lm, K, causal=(args.attn == "causal"), pool=pool)
    if args.method == "kd":
        with torch.no_grad(): model.head.load_state_dict(teacher.head.state_dict()); model.pos_emb.load_state_dict(teacher.pos_emb.state_dict()); model.dom_emb.load_state_dict(teacher.dom_emb.state_dict())
    before = {id(p) for p in model.parameters()}
    ctl = apply_method(model, args)
    mark_new_params_trainable(model, before)
    model.to(device)
    tot, trn = count_params(model)
    print(f"  params total={tot / 1e6:.2f}M trainable={trn / 1e6:.3f}M ({100 * trn / tot:.2f}%) | pool={pool} budget={budget} tokens/window")

    cw = np.ones(K)
    if args.cw_pow > 0:
        fr = np.maximum(cnt, 1) / cnt.sum(); cw = np.minimum((1.0 / fr) ** args.cw_pow, 1e9); cw = cw / (cw * fr).sum(); cw = np.minimum(cw, args.cw_max)
    cw_t = torch.tensor(cw, dtype=torch.float, device=device)
    print("  class weights:", {labels[i]: round(float(w), 2) for i, w in enumerate(cw)})

    decay, no_decay, head_p = [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (head_p if is_head_param(n) else decay if (p.ndim >= 2 and not n.startswith(("prompt", "prefix"))) else no_decay).append(p)
    groups = [dict(params=head_p, lr=args.lr_head, weight_decay=0.0), dict(params=no_decay, lr=base_lr, weight_decay=0.0),
              dict(params=decay, lr=base_lr, weight_decay=args.wd)]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    rng = random.Random(args.seed)
    meta = meta_of(args, lm_cfg, labels, model)

    def epoch_items():
        items = []
        for di, d in enumerate(train_docs):
            for lo, hi, a, b in make_windows(d["lens"], budget, args.train_core_frac, first_frac=rng.uniform(0.15, 1.0)):
                items.append((di, lo, hi, a, b))
        rng.shuffle(items); return items

    n_items = len(epoch_items())
    steps_per_epoch = max(1, n_items // args.bs)
    planned = int(args.epochs * steps_per_epoch)
    total, step, best_f1, best_step, t_bench = planned, 0, -1.0, 0, None
    eval_every = max(1, steps_per_epoch // max(1, args.eval_per_epoch))
    best_state = None
    print(f"  windows/epoch={n_items} steps/epoch={steps_per_epoch} planned steps={planned}")
    if args.method == "adalora":
        ctl = AdaLoRACtl(model, args.ada_init_r, args.ada_target_r, tinit=max(40, int(args.ada_tinit_frac * planned)),
                         tfinal=int(args.ada_tfinal_frac * planned), orth=args.ada_orth)
        print(f"  AdaLoRA: {len(ctl.mods)} modules, budget {ctl.init_budget} -> {ctl.target_budget} triplets, tinit={ctl.tinit} tfinal={ctl.tfinal}")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()

    def run_eval(tag):
        nonlocal best_f1, best_step, best_state
        if args.full_data:                      
            best_state, best_step = trainable_state(model), step; return
        lps = infer_docs(model, dev_docs, budget, args.infer_core_frac, device, amp)
        sc = score_docs(dev_docs, lps)
        print(f"  [eval {tag}] step={step} dev macro-F1={sc['F1']:.4f} P={sc['P']:.4f} R={sc['R']:.4f}", flush=True)
        if sc["F1"] > best_f1:
            best_f1, best_step, best_state = sc["F1"], step, trainable_state(model)
        model.train()

    model.train()
    t_start = time.time(); done = False; ep = 0; loss_ema = None
    while not done:
        order_w = epoch_items(); ep += 1
        for bi in range(0, len(order_w) - args.bs + 1, args.bs):
            if step == 5: t_bench = time.time()
            if step == 35 and t_bench is not None:                      
                sps = (time.time() - t_bench) / 30
                left = args.time_budget_min * 60 * 0.80 - (time.time() - t_global)
                total = max(60, min(planned, 35 + int(left / sps)))
                print(f"  speed={1 / sps:.2f} steps/s -> total steps={total} (planned {planned})", flush=True)
            if step >= total or (time.time() - t_global) > args.time_budget_min * 60 * 0.95: done = True; break
            lr_scale = lr_at(step, total, args.warmup, 1.0)
            for g in opt.param_groups: g["lr"] = g["base_lr"] * lr_scale
            with torch.autocast("cuda", enabled=amp):
                logits, lab = batch_forward(model, train_docs, order_w[bi:bi + args.bs], device)
            m = lab != IGN
            lf = logits.float()[m]; y = lab[m]
            ce = F.cross_entropy(lf, y, weight=cw_t)
            loss = ce
            if teacher is not None:
                ids_ = None
                with torch.no_grad(), torch.autocast("cuda", enabled=amp):
                    tl, _ = batch_forward(teacher, train_docs, order_w[bi:bi + args.bs], device)
                tl = tl.float()[m]; T = args.kd_T
                kd = F.kl_div(F.log_softmax(lf / T, -1), F.softmax(tl / T, -1), reduction="batchmean") * T * T
                loss = args.kd_alpha * ce + (1 - args.kd_alpha) * kd
            if ctl is not None: loss = loss + ctl.orth_loss()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            if ctl is not None: ctl.step(step + 1, total)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt); scaler.update(); step += 1
            loss_ema = loss.item() if loss_ema is None else 0.98 * loss_ema + 0.02 * loss.item()
            if step % 50 == 0:
                print(f"  step {step}/{total} ep {ep} loss {loss_ema:.4f} lr {base_lr * lr_scale:.2e} {(time.time() - t_start) / 60:.1f}min", flush=True)
            if step % eval_every == 0 and step < total and step >= 35: run_eval("mid")
    train_min = (time.time() - t_start) / 60
    run_eval("final")
    peak_gb = torch.cuda.max_memory_allocated() / 2 ** 30 if device.type == "cuda" else 0.0

    
    model.load_state_dict(best_state, strict=False)
    if args.method == "full":
        torch.save({"state": {k: v.detach().cpu() for k, v in model.state_dict().items()}, "meta": meta}, os.path.join(args.out, "best.pt"))
    elif best_state is not None:
        torch.save(best_state, os.path.join(args.out, "trainable.pt"))
    t0 = time.time()
    dev_lps = infer_docs(model, dev_docs, budget, args.infer_core_frac, device, amp)
    n_tok = sum(sum(d["lens"]) for d in dev_docs); inf_tps = n_tok / max(1e-6, time.time() - t0)
    save_lp(os.path.join(args.out, "dev"), dev_docs, dev_lps)
    sc = score_docs(dev_docs, dev_lps)
    per_dom = {}
    for dn, dm in SPLITS:
        sel = [i for i, d in enumerate(dev_docs) if d["dom"] == dm]
        if sel: per_dom[dn] = score_docs([dev_docs[i] for i in sel], [dev_lps[i] for i in sel])["F1"]
    print(f"  dev macro-F1={sc['F1']:.4f} P={sc['P']:.4f} R={sc['R']:.4f} | per-domain F1: {per_dom}")
    for i, l in enumerate(labels):
        print(f"    {l:24s} P={sc['per_p'][i]:.3f} R={sc['per_r'][i]:.3f} F1={sc['per_f1'][i]:.3f} support={sc['support'][i]}")
    with open(os.path.join(args.out, "dev_pred.json"), "w") as f: json.dump(decode(dev_lps, dev_docs, labels), f)

    res = dict(method=args.method, attn=args.attn, pool=pool, params_total=tot, params_trainable=trn, train_minutes=train_min, steps=step,
               planned_steps=planned, peak_gpu_gb=peak_gb, infer_tokens_per_s=inf_tps, best_step=best_step, full_data=args.full_data,
               dev_macro_F1=sc["F1"], dev_macro_P=sc["P"], dev_macro_R=sc["R"], dev_per_domain_F1=per_dom,
               per_label={l: dict(P=sc["per_p"][i], R=sc["per_r"][i], F1=sc["per_f1"][i], support=sc["support"][i]) for i, l in enumerate(labels)},
               args=vars(args), n_dev_docs=len(dev_docs))
    if ctl is not None and args.method == "adalora": res["adalora_rank"] = ctl.rank_report(); print("  AdaLoRA final ranks:", res["adalora_rank"])
    if not args.no_test:
        test_lps = infer_docs(model, test_docs, budget, args.infer_core_frac, device, amp)
        save_lp(os.path.join(args.out, "test"), test_docs, test_lps)
        pred = decode(test_lps, test_docs, labels)
        write_prediction(pred, test_docs, labels, args.out)
        res["n_test_docs"], res["n_test_sentences"] = len(pred), sum(len(v) for v in pred.values())
        print(f"  wrote {args.out}/submission.zip  ({res['n_test_sentences']} sentences in {len(pred)} docs)")
    with open(os.path.join(args.out, "results.json"), "w") as f: json.dump(res, f, indent=1, default=str)
    print(f"== done in {(time.time() - t_global) / 60:.1f} min | dev macro-F1 = {res['dev_macro_F1']:.4f}")


if __name__ == "__main__":
    main()

Writing rr.py


In [2]:
%%writefile rr_hyb.py

"""
rr_hyb.py -- CS781 Phase 3 (Rhetorical Roles) on top of YOUR Phase-1 *hybrid conv + sparse-attention* model (103M params).

Thin layer over rr.py (keep both files in the same folder): swaps in the hybrid trunk + its adaptation wiring and then calls rr.main(),
so windows, sentence pooling, class weights, KD, logging, log-prob dumps and ensemble3.py behave EXACTLY as for the decoder.

Layer pattern (12 layers, attn_every=4):  C C C A | C C C A | C C C A   (C = gated causal depthwise-conv mixer, A = global causal RoPE attention)
Methods here:  full | lora | lorma | adalora | ia3 | prompt | kd      (prefix tuning needs attention K/V in every layer -> not offered)
  python rr_hyb.py --data_dir DATA --tokenizer bpe_tokenizer.json --init model.pt --out runs/hyb_lorma --method lorma
The model is causal-only (causal convolutions + causal attention): no bidirectional option; sentences are pooled with the LAST token by default.
Default window = 1024 tokens (the model was trained at 1024).

Sections: A. hybrid model (copied from your notebook; one marked [P3-EDIT])  B. loader  C. wrapper / adaptation wiring  D. entry point
"""
import math
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import rr




class RMSNorm(nn.Module):
    """RMS normalization without mean subtraction."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return self.weight * x * torch.rsqrt(rms + self.eps)


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)          
    return torch.cat((-x2, x1), dim=-1)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)

    def forward(self, x, offset=0):
        T = x.size(-2)
        cos = self.cos[:, :, offset:offset + T, :].to(x.dtype)
        sin = self.sin[:, :, offset:offset + T, :].to(x.dtype)
        return x * cos + rotate_half(x) * sin


class GeGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down(F.gelu(self.gate(x)) * self.up(x)))


class GatedCausalConvMixer(nn.Module):
    """
    Local token mixer (Conformer / ConvBERT / gated-conv style) made causal for LM:
        x -> [value | gate] projection -> causal depthwise Conv1d(k) on value -> value * silu(gate) -> out proj
    Receptive field per layer = kernel size; no positional embedding needed (convolution is position-aware).
    Cost is O(T * k) instead of O(T^2).
    """
    def __init__(self, d_model, kernel_size, dropout):
        super().__init__()
        self.k = kernel_size
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.dw_weight = nn.Parameter(torch.empty(d_model, 1, kernel_size))
        nn.init.normal_(self.dw_weight, std=0.02)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, cache=None, offset=0):
        B, T, C = x.shape
        a, g = self.in_proj(x).chunk(2, dim=-1)
        a = a.transpose(1, 2)                                     
        if cache is None:
            state = a.new_zeros(B, C, self.k - 1)                 
        else:
            state = cache["state"]
        a_pad = torch.cat([state, a], dim=2)
        y = F.conv1d(a_pad, self.dw_weight.to(a.dtype), groups=C) 
        new_cache = {"state": a_pad[:, :, a_pad.size(2) - (self.k - 1):]}
        y = y.transpose(1, 2) * F.silu(g)
        return self.dropout(self.out_proj(y)), new_cache


class CausalSelfAttention(nn.Module):
    """Global causal multi-head attention with RoPE; used sparsely (every ATTN_EVERY-th layer)."""
    def __init__(self, d_model, n_heads, rope, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope, self.dropout = rope, dropout
        self.resid_dropout = nn.Dropout(dropout)

    def _split(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

    def forward(self, x, cache=None, offset=0):
        B, T, C = x.shape
        q, k, v = self._split(self.q_proj(x)), self._split(self.k_proj(x)), self._split(self.v_proj(x))
        q, k = self.rope(q, offset), self.rope(k, offset)
        if cache is None:                                         
            out = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        else:                                                     
            assert T == 1
            k, v = torch.cat([cache["k"], k], dim=2), torch.cat([cache["v"], v], dim=2)
            out = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out)), {"k": k, "v": v}


class HybridBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, rope, dropout, use_attention, conv_kernel):
        super().__init__()
        self.use_attention = use_attention
        self.norm_mix = RMSNorm(d_model)
        self.mixer = CausalSelfAttention(d_model, n_heads, rope, dropout) if use_attention \
            else GatedCausalConvMixer(d_model, conv_kernel, dropout)
        self.norm_ffn = RMSNorm(d_model)
        self.ffn = GeGLU(d_model, d_ff, dropout)

    def forward(self, x, cache=None, offset=0):
        h, new_cache = self.mixer(self.norm_mix(x), cache=cache, offset=offset)
        x = x + h
        x = x + self.ffn(self.norm_ffn(x))
        return x, new_cache


class HybridConvAttnLM(nn.Module):
    """
    Encoder-style (conformer/ConvBERT-like) hybrid adapted to causal language modelling:
    layer pattern for attn_every=4, 12 layers:  C C C A | C C C A | C C C A
    (C = gated causal depthwise-conv mixer, A = global causal RoPE attention), GeGLU FFN everywhere.
    """
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len,
                 attn_every=4, conv_kernel=15, dropout=0.1, rope_base=10000.0, tie_weights=True,
                 attn_layers=None):                                    
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.emb_dropout = nn.Dropout(dropout)
        self.rope = RotaryEmbedding(d_model // n_heads, max_seq_len, rope_base)
        self.blocks = nn.ModuleList([
            HybridBlock(d_model, n_heads, d_ff, self.rope, dropout,
                        use_attention=(bool(attn_layers[i]) if attn_layers is not None else (i + 1) % attn_every == 0), conv_kernel=conv_kernel)
            for i in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_weights:
            self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _run(self, idx, caches=None, offset=0):
        B, T = idx.shape
        if offset + T > self.max_seq_len:
            raise ValueError(f"Sequence length {offset + T} exceeds max_seq_len={self.max_seq_len}")
        x = self.emb_dropout(self.token_emb(idx))
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, c = blk(x, cache=None if caches is None else caches[i], offset=offset)
            new_caches.append(c)
        return self.lm_head(self.final_norm(x)), new_caches

    def forward(self, idx, targets=None):
        logits, _ = self._run(idx)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss





DEFAULT_HYB_CFG = dict(vocab_size=12000, d_model=768, n_layers=12, n_heads=12, d_ff=2560, max_seq_len=1024,
                       attn_every=4, conv_kernel=15, dropout=0.0, rope_base=10000.0)


def build_lm_hyb(cfg):
    c = {**DEFAULT_HYB_CFG, **{k: v for k, v in cfg.items() if k in DEFAULT_HYB_CFG or k == "attn_layers"}}
    if c.get("attn_layers") is None: c["attn_layers"] = [(i + 1) % c["attn_every"] == 0 for i in range(c["n_layers"])]
    c["attn_layers"] = [bool(x) for x in c["attn_layers"]]
    assert len(c["attn_layers"]) == c["n_layers"]
    return HybridConvAttnLM(**c, tie_weights=True), c


def load_phase1_hyb(path, n_layers=None, layer_map=None):
    """Accepts model.pt ({state_dict, config}), checkpoint.pt ({model_state, run_config}), model_stepXXXX.pt ({step, state_dict}) or a raw state_dict.
    The layer types (conv / attention) are read from the weights themselves."""
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = {}
    if isinstance(ck, dict) and "state_dict" in ck:
        sd, cfg = ck["state_dict"], dict(ck.get("config", {}))
    elif isinstance(ck, dict) and "model_state" in ck:
        sd = ck["model_state"]; rc = ck.get("run_config", {}) or {}
        cfg = {k: v for k, v in dict(n_heads=rc.get("N_HEADS"), attn_every=rc.get("ATTN_EVERY"), conv_kernel=rc.get("CONV_KERNEL"),
                                     rope_base=rc.get("ROPE_BASE"), max_seq_len=rc.get("BLOCK_SIZE")).items() if v is not None}
    else:
        sd = ck
    sd = {k.replace("module.", "", 1) if k.startswith("module.") else k: v for k, v in sd.items()}
    L = 1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks."))
    is_attn = [f"blocks.{i}.mixer.q_proj.weight" in sd for i in range(L)]
    cfg["vocab_size"], cfg["d_model"] = sd["token_emb.weight"].shape
    cfg["d_ff"] = sd["blocks.0.ffn.gate.weight"].shape[0]
    conv = [k for k in sd if k.endswith("mixer.dw_weight")]
    if conv: cfg["conv_kernel"] = sd[conv[0]].shape[-1]
    cfg["n_layers"], cfg["attn_layers"] = L, is_attn
    if layer_map is not None:                                                
        new = {k: v for k, v in sd.items() if not k.startswith("blocks.")}
        for ni, oi in enumerate(layer_map):
            for k, v in sd.items():
                if k.startswith(f"blocks.{oi}."):
                    new[f"blocks.{ni}." + k.split(".", 2)[2]] = v
        sd, cfg["n_layers"], cfg["attn_layers"] = new, len(layer_map), [is_attn[o] for o in layer_map]
    lm, cfg = build_lm_hyb(cfg)
    missing, unexpected = lm.load_state_dict(sd, strict=False)
    bad = [k for k in missing if k != "lm_head.weight"] + list(unexpected)
    assert not bad, f"checkpoint/architecture mismatch: {bad[:6]}"
    return lm, cfg





class HybRR(rr.RRModel):
    """Hybrid trunk + the same sentence pooling / head as the decoder.  Causal only; right padding cannot leak into real tokens."""
    def __init__(self, lm, K, causal=True, pool="last", head_dropout=0.1, pos_dim=32, dom_dim=8):
        super().__init__(lm, K, causal=True, pool=pool, head_dropout=head_dropout, pos_dim=pos_dim, dom_dim=dom_dim)
        self.emb_dropout = lm.emb_dropout

    def set_causal(self, causal):
        self.causal = True

    def trunk(self, ids, pad_mask):
        B = ids.size(0)
        x = self.emb_dropout(self.token_emb(ids)); P = 0
        if self.prompt is not None:                                           
            P = self.prompt.size(0)
            x = torch.cat([self.prompt.unsqueeze(0).expand(B, -1, -1).to(x.dtype), x], 1)
        for b in self.blocks:
            x, _ = b(x)
        return self.final_norm(x)[:, P:]


MIX_T = ("q_proj", "k_proj", "v_proj", "in_proj", "out_proj")     
FFN_T = ("gate", "up", "down")                                    
_DEC_TARGETS = "q_proj,k_proj,v_proj,out_proj,gate,up,down"
_DEC_IA3 = "k_proj,v_proj,down"


def apply_method_hyb(model, args):
    m = args.method
    for p in model.parameters(): p.requires_grad = False
    for n, p in model.named_parameters():
        if rr.is_head_param(n): p.requires_grad = True
    targets = args.targets
    if targets == _DEC_TARGETS: targets += ",in_proj"                        
    tg = [t.strip() for t in targets.split(",") if t.strip()]
    ia3_t = args.ia3_targets + (",in_proj" if args.ia3_targets == _DEC_IA3 else "")

    def parent_of(b, t): return b.mixer if t in MIX_T else b.ffn

    def wrap_linears(factory, names):
        for b in model.blocks:
            for t in names:
                par = parent_of(b, t)
                if hasattr(par, t): setattr(par, t, factory(getattr(par, t)))   

    ctl = None
    if m in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    elif m == "lora":
        wrap_linears(lambda l: rr.LoRALinear(l, args.rank, args.alpha, args.lora_dropout), tg)
    elif m == "lorma":
        wrap_linears(lambda l: rr.LoRMAPlusLinear(l, args.rank, args.alpha), tg)
    elif m == "adalora":
        wrap_linears(lambda l: rr.AdaLoRALinear(l, args.ada_init_r, args.alpha_ada, args.lora_dropout), tg)
        ctl = "pending"
    elif m == "ia3":                                                         
        for b in model.blocks:
            for t in [x.strip() for x in ia3_t.split(",") if x.strip()]:
                par = parent_of(b, t)
                if hasattr(par, t): setattr(par, t, rr.IA3In(getattr(par, t)) if t == "down" else rr.IA3Out(getattr(par, t)))
    elif m == "prompt":
        ids = torch.randint(260, model.token_emb.num_embeddings, (args.n_prompt,))
        model.prompt = nn.Parameter(model.token_emb.weight[ids].detach().clone())
    else:
        raise ValueError(f"hybrid pipeline supports full | lora | lorma | adalora | ia3 | prompt | kd, got {m}")
    return ctl





def _patched():
    rr.load_phase1, rr.build_lm = load_phase1_hyb, build_lm_hyb
    rr.RRModel, rr.apply_method = HybRR, apply_method_hyb                    


if __name__ == "__main__":
    _patched()
    argv = sys.argv
    if "--window" not in argv: argv += ["--window", "1024"]
    if "--attn" in argv: argv[argv.index("--attn") + 1] = "causal"           
    else: argv += ["--attn", "causal"]
    sys.argv = argv
    rr.PAD_MULTIPLE = 128                                                    
    torch.backends.cudnn.benchmark = True                                    
    rr.main()

Writing rr_hyb.py


In [3]:
%%writefile ensemble3.py

"""
ensemble3.py -- combine Phase-3 runs (each run dir holds dev_lp.npz/dev_ids.json/test_lp.npz/test_ids.json written by rr.py).

  python ensemble3.py --runs_dir runs --data_dir data --names lora,lorma,adalora --out subs/ens_peft
  python ensemble3.py --runs_dir runs --data_dir data --all --greedy --out subs/ens_greedy        # forward selection on dev
  add  --bias_tune  to learn one additive log-prob bias per class on dev (coordinate ascent on macro-F1)

Everything that is *selected on dev* (greedy subset, class biases) is optimistically scored on dev -- the printed number is
NOT an unbiased estimate; the Codabench leaderboard is.
"""
import argparse, glob, json, os, sys
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import rr


def load_run(d, kind):
    z = np.load(os.path.join(d, f"{kind}_lp.npz"))
    ids = json.load(open(os.path.join(d, f"{kind}_ids.json")))
    lens = z["lens"]; lp = z["lp"]; assert len(ids) == len(lens) and lens.sum() == len(lp)
    return ids, lens, lp


def combine(lps, w=None, mode="prob"):
    w = np.ones(len(lps)) if w is None else np.asarray(w, float)
    w = w / w.sum()
    if mode == "prob":
        return np.log(np.maximum(sum(wi * np.exp(l.astype(np.float64)) for wi, l in zip(w, lps)), 1e-12))
    return sum(wi * l.astype(np.float64) for wi, l in zip(w, lps))


def macro_f1(y, lp, bias=None):
    p = (lp + (0 if bias is None else bias)).argmax(1)
    return rr.macro_prf(y, p, lp.shape[1])["F1"]


def tune_bias(y, lp, grid=np.linspace(-2, 2, 21), rounds=3):
    K = lp.shape[1]; b = np.zeros(K); best = macro_f1(y, lp, b)
    for _ in range(rounds):
        for c in range(K):
            cand = []
            for g in grid:
                bb = b.copy(); bb[c] = g; cand.append((macro_f1(y, lp, bb), -abs(g), g))
            f, _, g = max(cand)
            if f > best + 1e-9: best, b[c] = f, g
    return b, best


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs_dir", required=True); ap.add_argument("--data_dir", required=True); ap.add_argument("--out", required=True)
    ap.add_argument("--names", default=""); ap.add_argument("--all", action="store_true")
    ap.add_argument("--weights", default="", help="comma list, same order as --names")
    ap.add_argument("--mode", default="prob", choices=["prob", "logp"])
    ap.add_argument("--greedy", action="store_true", help="forward selection (with replacement) of members by dev macro-F1")
    ap.add_argument("--greedy_steps", type=int, default=10)
    ap.add_argument("--bias_tune", action="store_true")
    args = ap.parse_args()

    labels, label2idx = rr.load_vocab(rr.load_json(os.path.join(args.data_dir, "label_vocab.json")))
    names = sorted(n for n in (os.path.basename(os.path.dirname(p)) for p in glob.glob(os.path.join(args.runs_dir, "*", "dev_lp.npz"))) if not n.startswith(("smoke", "tmp"))) if args.all \
        else [n for n in args.names.split(",") if n]
    assert names, "give --names a,b,c or --all"
    dev, test = {}, {}
    ref_dev = ref_test = None
    for n in names:
        d = os.path.join(args.runs_dir, n)
        if not os.path.exists(os.path.join(d, "dev_lp.npz")): print(f"skip {n}: no dev_lp.npz"); continue
        dev[n] = load_run(d, "dev")
        if ref_dev is None: ref_dev = dev[n][0]
        assert dev[n][0] == ref_dev, f"{n}: dev doc order differs"
        if os.path.exists(os.path.join(d, "test_lp.npz")):
            test[n] = load_run(d, "test")
            if ref_test is None: ref_test = test[n][0]
            assert test[n][0] == ref_test, f"{n}: test doc order differs"
    names = [n for n in names if n in dev]
    
    gold = {}
    for dn in ("CL", "IT"):
        for did, sents, labs in rr.iter_raw_docs(rr.load_json(os.path.join(args.data_dir, f"{dn}_dev.json"))):
            gold[did] = np.array([rr.label_to_idx(l, label2idx, labels) for l in labs])
    y = np.concatenate([gold[i] for i in ref_dev])
    assert len(y) == len(dev[names[0]][2])
    print("\nsingle runs (dev macro-F1):")
    for n in names: print(f"  {n:28s} {macro_f1(y, dev[n][2]):.4f}")

    if args.greedy:
        chosen, cur = [], None
        for s in range(args.greedy_steps):
            best = None
            for n in names:
                f = macro_f1(y, combine([dev[m][2] for m in chosen + [n]], mode=args.mode))
                if best is None or f > best[0]: best = (f, n)
            if cur is not None and best[0] <= cur + 1e-6: break
            chosen.append(best[1]); cur = best[0]; print(f"  greedy step {s + 1}: +{best[1]} -> {cur:.4f}")
        members, w = chosen, None
    else:
        members = names
        w = [float(x) for x in args.weights.split(",")] if args.weights else None
        if w: assert len(w) == len(members)
    
    dev_lp = combine([dev[m][2] for m in members], w, args.mode)
    f_plain = macro_f1(y, dev_lp)
    print(f"\nensemble of {members}\n  dev macro-F1 = {f_plain:.4f}" + ("  (members chosen ON dev -> optimistic)" if args.greedy else ""))
    bias = None
    if args.bias_tune:
        bias, fb = tune_bias(y, dev_lp)
        print(f"  with per-class bias tuned on dev: {fb:.4f}  (OPTIMISTIC, tuned on the same data)  bias={dict(zip(labels, np.round(bias, 2).tolist()))}")
    sc = rr.macro_prf(y, (dev_lp + (0 if bias is None else bias)).argmax(1), len(labels))
    for i, l in enumerate(labels): print(f"    {l:24s} P={sc['per_p'][i]:.3f} R={sc['per_r'][i]:.3f} F1={sc['per_f1'][i]:.3f} support={sc['support'][i]}")

    if not test or not all(m in test for m in members):
        print("no test log-probs for all members -> not writing a submission"); return
    t_lp = combine([test[m][2] for m in members], w, args.mode) + (0 if bias is None else bias)
    ids, lens = test[members[0]][0], test[members[0]][1]
    off = np.concatenate([[0], np.cumsum(lens)])
    pred = {did: [labels[k] for k in t_lp[off[i]:off[i + 1]].argmax(1)] for i, did in enumerate(ids)}
    
    n_sent = {}
    for dn in ("CL", "IT"):
        for did, sents, _ in rr.iter_raw_docs(rr.load_json(os.path.join(args.data_dir, f"{dn}_test.json"))): n_sent[did] = len(sents)
    assert set(pred) == set(n_sent), "ids differ from the test files"
    assert all(len(pred[i]) == n_sent[i] for i in pred), "sentence counts differ from the test files"
    os.makedirs(args.out, exist_ok=True)
    pj = os.path.join(args.out, "prediction.json"); json.dump(pred, open(pj, "w"))
    import zipfile
    with zipfile.ZipFile(os.path.join(args.out, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
    json.dump(dict(members=members, weights=w, mode=args.mode, bias_tune=bool(args.bias_tune), dev_F1=sc["F1"]), open(os.path.join(args.out, "ensemble.json"), "w"))
    print(f"wrote {args.out}/submission.zip  ({len(pred)} docs, {sum(map(len, pred.values()))} sentences)")


if __name__ == "__main__":
    main()

Writing ensemble3.py


In [4]:
%%writefile evaluate_phase3.py

"""Local evaluator for Phase 3.

The competition scorer evaluates two gold/reference JSON files together and
one participant prediction.json. This local evaluator mirrors that setup
using the student's own held-out splits.

Usage:
  python evaluate_phase3.py \
      --truth-cl cl_test_truth.json \
      --truth-it it_test_truth.json \
      --prediction prediction.json \
      --labels labels.json

Each truth file must be a JSON object mapping document IDs to ordered lists of
sentence labels. prediction.json must have the same structure and contain
predictions for documents from both reference files.

labels.json may be either:
  ["Label1", "Label2", ...]
or a label->id / id->label mapping.
"""
import argparse
import json
import sys

from sklearn.metrics import f1_score, precision_score, recall_score

EXPECTED_LABEL_COUNT = 13


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def normalize_id(doc_id):
    if not isinstance(doc_id, str) or not doc_id.strip():
        raise ValueError("Document IDs must be non-empty strings.")
    return doc_id[:-4] if doc_id.endswith(".txt") else doc_id


def load_vocab(data):
    if isinstance(data, list):
        labels = data
    elif isinstance(data, dict) and data:
        values = list(data.values())
        if all(isinstance(v, int) and not isinstance(v, bool) for v in values):
            
            labels = list(data.keys())
        elif all(isinstance(v, str) for v in values):
            
            labels = values
        else:
            raise ValueError("Unsupported labels.json format.")
    else:
        raise ValueError("labels.json must be a non-empty list or object.")

    if len(labels) != EXPECTED_LABEL_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_LABEL_COUNT} labels, found {len(labels)}."
        )
    if any(not isinstance(label, str) or not label.strip() for label in labels):
        raise ValueError("Labels must be unique non-empty strings.")
    if len(set(labels)) != len(labels):
        raise ValueError("Labels must be unique non-empty strings.")
    return labels


def normalize_reference(data, source_name):
    if not isinstance(data, dict):
        raise ValueError(f"{source_name}: reference data must be a JSON object.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"{source_name}: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def normalize_predictions(data):
    if not isinstance(data, dict):
        raise ValueError("prediction.json must be a JSON object mapping document IDs to label lists.")

    normalized = {}
    for raw_id, labels in data.items():
        doc_id = normalize_id(raw_id)
        if doc_id in normalized:
            raise ValueError(
                f"prediction.json: duplicate document ID after .txt normalization: {doc_id}"
            )
        normalized[doc_id] = labels
    return normalized


def main():
    parser = argparse.ArgumentParser(
        description=(
            "Evaluate Phase 3 sentence-label predictions on two local held-out "
            "reference splits, mirroring the competition's combined scoring setup."
        )
    )
    parser.add_argument(
        "--truth-cl",
        required=True,
        help="Local held-out reference JSON for the CL split.",
    )
    parser.add_argument(
        "--truth-it",
        required=True,
        help="Local held-out reference JSON for the IT split.",
    )
    parser.add_argument(
        "--prediction",
        required=True,
        help="Participant prediction JSON covering documents from both splits.",
    )
    parser.add_argument(
        "--labels",
        required=True,
        help="Public Phase 3 label vocabulary JSON.",
    )
    args = parser.parse_args()

    truth_cl = normalize_reference(load_json(args.truth_cl), "--truth-cl")
    truth_it = normalize_reference(load_json(args.truth_it), "--truth-it")
    predictions = normalize_predictions(load_json(args.prediction))
    labels = load_vocab(load_json(args.labels))
    allowed = set(labels)

    overlap = set(truth_cl) & set(truth_it)
    if overlap:
        sample = sorted(overlap)[:5]
        suffix = "..." if len(overlap) > 5 else ""
        raise ValueError(
            "The two reference splits contain overlapping document IDs: "
            + ", ".join(sample)
            + suffix
        )

    truth = {}
    truth.update(truth_cl)
    truth.update(truth_it)

    if set(truth) != set(predictions):
        missing = sorted(set(truth) - set(predictions))
        extra = sorted(set(predictions) - set(truth))
        parts = [
            f"missing={len(missing)}",
            f"extra={len(extra)}",
        ]
        if missing:
            parts.append(f"example missing ID(s)={missing[:5]}")
        if extra:
            parts.append(f"example extra ID(s)={extra[:5]}")
        raise ValueError("ID mismatch: " + "; ".join(parts))

    y_true = []
    y_pred = []

    for doc_id in truth:
        gold = truth[doc_id]
        pred = predictions[doc_id]

        if not isinstance(gold, list) or not isinstance(pred, list):
            raise ValueError(f"{doc_id}: labels must be JSON lists.")
        if len(gold) != len(pred):
            raise ValueError(
                f"{doc_id}: sentence-count mismatch ({len(gold)} vs {len(pred)})."
            )
        if any(label not in allowed for label in gold + pred):
            raise ValueError(f"{doc_id}: unknown label encountered.")

        y_true.extend(gold)
        y_pred.extend(pred)

    score = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_precision = precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    macro_recall = recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)

    print(f"CL documents: {len(truth_cl)}")
    print(f"IT documents: {len(truth_it)}")
    print(f"Total documents: {len(truth)}")
    print(f"Total sentences: {len(y_true)}")
    print(f"Macro-F1:        {score:.6f}")
    print(f"Macro-Precision: {macro_precision:.6f}")
    print(f"Macro-Recall:    {macro_recall:.6f}")


if __name__ == "__main__":
    main()


Writing evaluate_phase3.py


## 1. Environment, Phase-1 files

In [5]:
import os, sys, json, time, glob, subprocess, shutil, collections
import numpy as np, pandas as pd
WORK = "/kaggle/working"; sys.path.insert(0, WORK)
def find(name, root="/kaggle/input"):
    hits = [os.path.join(d, name) for d, _, fs in os.walk(root, followlinks=True) if name in fs]
    return sorted(hits, key=os.path.getmtime)
cands = [c for n in ("model.pt", "checkpoint.pt") for c in reversed(find(n)) if "hybrid" in c.lower()]      
cands += [c for c in glob.glob("/kaggle/input/**/snapshots/model_step*.pt", recursive=True) if "hybrid" in c.lower()]
print("hybrid candidates:", cands)
INIT = cands[0] if cands else None                       
assert INIT, f"attach the hybrid Phase-1 output (path must contain 'hybrid'). /kaggle/input has: {os.listdir('/kaggle/input')}"
toks = find("bpe_tokenizer.json"); TOK = next((t for t in toks if os.path.dirname(t) == os.path.dirname(INIT)), (toks or [None])[-1])
assert TOK, "bpe_tokenizer.json not found - attach it (it must be the SAME tokenizer the model was trained with)"
print("INIT:", INIT, "\nTOK :", TOK)
import torch; print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

hybrid candidates: ['/kaggle/input/notebooks/luckyj12/hybrid/cs781_phase1/hybrid_conv_attn_train_supreme_court-train_acts-train_high_courts-train_tribunals/model.pt', '/kaggle/input/notebooks/luckyj12/hybrid/cs781_phase1/hybrid_conv_attn_train_supreme_court-train_acts-train_high_courts-train_tribunals/checkpoint.pt', '/kaggle/input/notebooks/luckyj12/hybrid-part-2/cs781_phase1/hybrid_conv_attn_train_supreme_court-train_acts-train_high_courts-train_tribunals/checkpoint.pt', '/kaggle/input/notebooks/luckyj12/hybrid-part-2/cs781_phase1/hybrid_conv_attn_train_supreme_court-train_acts-train_high_courts-train_tribunals/snapshots/model_step7500.pt']
INIT: /kaggle/input/notebooks/luckyj12/hybrid/cs781_phase1/hybrid_conv_attn_train_supreme_court-train_acts-train_high_courts-train_tribunals/model.pt 
TOK : /kaggle/input/notebooks/luckyj12/hybrid/cs781_phase1/hybrid_conv_attn_train_supreme_court-train_acts-train_high_courts-train_tribunals/bpe_tokenizer.json
GPUs: 2 ['Tesla T4', 'Tesla T4']


## 2. Data (Hugging Face `Exploration-Lab/CS-781-Capstone`)

In [6]:
from huggingface_hub import hf_hub_download, list_repo_files
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HuggingFace")
DATA = f"{WORK}/p3data"; os.makedirs(DATA, exist_ok=True)
repo_files = [f for f in list_repo_files("Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=hf_token) if "Phase_3" in f]
print("\n".join(repo_files))
WANT = ["CL_train.json", "CL_dev.json", "IT_train.json", "IT_dev.json", "CL_test.json", "IT_test.json", "label_vocab.json"]
for w in WANT:
    hit = [f for f in repo_files if os.path.basename(f) == w]
    assert hit, f"{w} not found in the repo listing above"
    if not os.path.exists(f"{DATA}/{w}"):
        shutil.copy(hf_hub_download("Exploration-Lab/CS-781-Capstone", hit[0], repo_type="dataset", token=hf_token), f"{DATA}/{w}")
print(sorted(os.listdir(DATA)))

Phase_3_RR/Test/CL_test.json
Phase_3_RR/Test/IT_test.json
Phase_3_RR/Train/CL_dev.json
Phase_3_RR/Train/CL_train.json
Phase_3_RR/Train/IT_dev.json
Phase_3_RR/Train/IT_train.json
Phase_3_RR/Train/label_vocab.json


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

['CL_dev.json', 'CL_test.json', 'CL_train.json', 'IT_dev.json', 'IT_test.json', 'IT_train.json', 'label_vocab.json']


## 3. Data analysis (material for the report) + schema check

In [7]:
import rr
from collections import Counter
labels, label2idx = rr.load_vocab(rr.load_json(f"{DATA}/label_vocab.json")); print(labels)
raw = {k: list(rr.iter_raw_docs(rr.load_json(f"{DATA}/{k}.json"))) for k in ["CL_train","CL_dev","IT_train","IT_dev","CL_test","IT_test"]}
first = rr.load_json(f"{DATA}/CL_train.json")
print("top-level type:", type(first).__name__, "| first item keys:", list(first[0].keys()) if isinstance(first, list) else list(next(iter(first.values())).keys()) if isinstance(next(iter(first.values())), dict) else "id -> list")
rows = []
for k, docs in raw.items():
    n_s = [len(s) for _, s, _ in docs]
    rows.append(dict(split=k, docs=len(docs), sentences=sum(n_s), mean_sent_per_doc=round(np.mean(n_s), 1), max_sent_per_doc=max(n_s),
                     has_labels=docs[0][2] is not None, mean_words_per_sentence=round(np.mean([len(x.split()) for _, s, _ in docs for x in s]), 1)))
display(pd.DataFrame(rows))
dist = pd.DataFrame({k: Counter(l if isinstance(l, str) else labels[l] for _, _, ls in raw[k] for l in ls) for k in ["CL_train","CL_dev","IT_train","IT_dev"]}).reindex(labels).fillna(0).astype(int)
dist["train_total"] = dist.CL_train + dist.IT_train; dist["train_%"] = (100 * dist.train_total / dist.train_total.sum()).round(1)
display(dist)

for dn in ("CL", "IT"):
    json.dump({did: [l if isinstance(l, str) else labels[l] for l in ls] for did, _, ls in raw[f"{dn}_dev"]}, open(f"{DATA}/truth_{dn}_dev.json", "w"))
assert not set(d for d, _, _ in raw["CL_dev"]) & set(d for d, _, _ in raw["IT_dev"]), "CL/IT dev ids overlap"

['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
top-level type: list | first item keys: ['id', 'text', 'labels']


,split,docs,sentences,mean_sent_per_doc,max_sent_per_doc,has_labels,mean_words_per_sentence
0,CL_train,40,10593,264.8,1816,True,32.7
1,CL_dev,5,1396,279.2,848,True,33.7
2,IT_train,40,6252,156.3,442,True,35.3
3,IT_dev,5,746,149.2,382,True,32.3
4,CL_test,5,1339,267.8,942,False,33.2
5,IT_test,5,858,171.6,420,False,37.6


,CL_train,CL_dev,IT_train,IT_dev,train_total,train_%
Fact,2979,249,1119,195,4098,24.3
Issue,17,3,90,12,107,0.6
ArgumentPetitioner,976,344,637,56,1613,9.6
ArgumentRespondent,996,236,352,36,1348,8.0
Statute,367,58,732,62,1099,6.5
Dissent,284,0,0,0,284,1.7
PrecedentReliedUpon,1305,142,640,112,1945,11.5
PrecedentNotReliedUpon,209,0,360,50,569,3.4
PrecedentOverruled,0,0,3,0,3,0.0
RulingByLowerCourt,438,56,436,46,874,5.2


## 4. Launcher + smoke test (≈ 5–10 min)
One process per GPU (`CUDA_VISIBLE_DEVICES`). The smoke test runs **every** method on ~10 documents for one epoch so a typo costs minutes, not hours. If it passes, `smoke/` can be ignored.

In [8]:
os.makedirs(f"{WORK}/runs", exist_ok=True)
BUDGET = 60   

def cmd(name, gpu, extra, epochs, budget=None, teacher_run=None):
    out = f"{WORK}/runs/{name}"; os.makedirs(out, exist_ok=True)
    t = f"--teacher {WORK}/runs/{teacher_run}/best.pt" if teacher_run else ""
    return (f"PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES={gpu} python rr_hyb.py --data_dir {DATA} --tokenizer {TOK} --init {INIT} --out {out} "
            f"--epochs {epochs} --time_budget_min {budget or BUDGET} {t} {extra} > {out}/log.txt 2>&1")

def launch(gpu, jobs):
    """jobs = [(name, extra_args, epochs, teacher_run_or_None)] run one after another on `gpu`."""
    p = subprocess.Popen(" ; ".join(cmd(n, gpu, e, ep, teacher_run=t) for n, e, ep, t in jobs), shell=True)
    return p, [j[0] for j in jobs]

def wait(procs, every=120):
    while any(p.poll() is None for p, _ in procs):
        time.sleep(every)
        for p, names in procs:
            logs = [l for n in names for l in glob.glob(f"{WORK}/runs/{n}/log.txt")]
            if not logs: continue
            lg = max(logs, key=os.path.getmtime); tail = open(lg).read().strip().splitlines()[-1:]
            print(f"[{lg.split('/')[-2]}] {'running' if p.poll() is None else 'done'} | {tail[0][:130] if tail else ''}")
    for p, names in procs:
        for n in names:
            ok = os.path.exists(f"{WORK}/runs/{n}/results.json")
            print(f"== {n}: {'OK' if ok else 'FAILED  <-- see runs/' + n + '/log.txt'}")

def table(pattern="*"):
    rows = []
    for d in sorted(glob.glob(f"{WORK}/runs/{pattern}/results.json")):
        x = json.load(open(d)); pd_ = x.get("dev_per_domain_F1", {})
        rows.append(dict(run=d.split("/")[-2], method=x["method"], attn=x["attn"], dev_F1=round(x["dev_macro_F1"], 4), P=round(x["dev_macro_P"], 4), R=round(x["dev_macro_R"], 4),
                         CL=round(pd_.get("CL", float("nan")), 4), IT=round(pd_.get("IT", float("nan")), 4), trainable_M=round(x["params_trainable"] / 1e6, 3),
                         train_min=round(x["train_minutes"], 1), steps=f"{x['steps']}/{x['planned_steps']}", best_step=x["best_step"], gpu_GB=round(x["peak_gpu_gb"], 1),
                         infer_tok_s=int(x["infer_tokens_per_s"])))
    return pd.DataFrame(rows).sort_values("dev_F1", ascending=False) if rows else pd.DataFrame()

def move_smoke():
    os.makedirs(f"{WORK}/smoke", exist_ok=True)
    for d in glob.glob(f"{WORK}/runs/smoke_*"): shutil.move(d, f"{WORK}/smoke/" + os.path.basename(d))

SMOKE = "--max_train_docs 10 --max_dev_docs 4 --no_test"
pA = launch(0, [("smoke_full", f"--method full {SMOKE}", 1, None), ("smoke_kd", f"--method kd {SMOKE}", 1, "smoke_full"), ("smoke_lora", f"--method lora {SMOKE}", 1, None), ("smoke_adalora", f"--method adalora {SMOKE}", 1, None)])
pB = launch(1, [("smoke_lorma", f"--method lorma {SMOKE}", 1, None), ("smoke_ia3", f"--method ia3 {SMOKE}", 1, None), ("smoke_prompt", f"--method prompt {SMOKE}", 1, None)])
wait([pA, pB], every=30)
display(table("smoke_*"))
print(open(f"{WORK}/runs/smoke_lorma/log.txt").read()[-1800:])
move_smoke()      

[smoke_full] running |   [eval final] step=7 dev macro-F1=0.0505 P=0.0625 R=0.0968
[smoke_lorma] running |   [eval final] step=7 dev macro-F1=0.0475 P=0.0625 R=0.0985
[smoke_lora] running |   windows/epoch=62 steps/epoch=7 planned steps=7
[smoke_prompt] running |   class weights: {'Fact': 0.62, 'Issue': 2.58, 'ArgumentPetitioner': 0.99, 'ArgumentRespondent': 1.17, 'Statute': 1.34, 'Dissent':
[smoke_adalora] running |   AdaLoRA: 66 modules, budget 1584 -> 1056 triplets, tinit=40 tfinal=2
[smoke_prompt] done | == done in 0.3 min | dev macro-F1 = 0.0238
[smoke_adalora] done | == done in 0.3 min | dev macro-F1 = 0.0304
[smoke_prompt] done | == done in 0.3 min | dev macro-F1 = 0.0238
== smoke_full: OK
== smoke_kd: OK
== smoke_lora: OK
== smoke_adalora: OK
== smoke_lorma: OK
== smoke_ia3: OK
== smoke_prompt: OK


,run,method,attn,dev_F1,P,R,CL,IT,trainable_M,train_min,steps,best_step,gpu_GB,infer_tok_s
3,smoke_kd,kd,causal,0.0538,0.0478,0.0912,0.0446,0.0883,56.469,0.1,7/7,7,4.0,53443
1,smoke_full,full,causal,0.0505,0.0625,0.0968,0.0441,0.0503,103.132,0.1,7/7,7,6.3,28158
5,smoke_lorma,lorma,causal,0.0475,0.0625,0.0985,0.0409,0.0482,3.231,0.1,7/7,7,5.6,21530
4,smoke_lora,lora,causal,0.0396,0.0582,0.0901,0.0315,0.0448,2.776,0.1,7/7,7,5.7,19877
2,smoke_ia3,ia3,causal,0.0329,0.0555,0.0892,0.0230,0.0468,0.060,0.1,7/7,7,5.3,22983
0,smoke_adalora,adalora,causal,0.0304,0.0493,0.0877,0.0208,0.0423,4.160,0.1,7/7,7,5.8,18762
6,smoke_prompt,prompt,causal,0.0238,0.0403,0.0751,0.0231,0.0249,0.036,0.1,7/7,7,4.1,25474


Fact': 227, 'Issue': 13, 'ArgumentPetitioner': 88, 'ArgumentRespondent': 63, 'Statute': 48, 'Dissent': 0, 'PrecedentReliedUpon': 60, 'PrecedentNotReliedUpon': 12, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 38, 'RatioOfTheDecision': 198, 'RulingByPresentCourt': 51, 'None': 21}
  params total=106.35M trainable=3.231M (3.04%) | pool=last budget=1024 tokens/window
  class weights: {'Fact': 0.62, 'Issue': 2.58, 'ArgumentPetitioner': 0.99, 'ArgumentRespondent': 1.17, 'Statute': 1.34, 'Dissent': 9.3, 'PrecedentReliedUpon': 1.2, 'PrecedentNotReliedUpon': 2.68, 'PrecedentOverruled': 9.3, 'RulingByLowerCourt': 1.51, 'RatioOfTheDecision': 0.66, 'RulingByPresentCourt': 1.3, 'None': 2.03}
  windows/epoch=62 steps/epoch=7 planned steps=7
  [eval final] step=7 dev macro-F1=0.0475 P=0.0625 R=0.0985
  dev macro-F1=0.0475 P=0.0625 R=0.0985 | per-domain F1: {'CL': 0.04087171355783832, 'IT': 0.04821020563594822}
    Fact                     P=0.202 R=0.088 F1=0.123 support=204
    Issue               

## 5. YOUR KNOBS
Untuned starting values. **The course requires that you choose / tune these yourself and explain why** – change them, rerun a round, compare in the table, and write the analysis in your report.
`EP` = epochs, `LR` = learning rate of the adapter parameters (`None` → script default; the head always uses `--lr_head 1e-3`).

In [9]:
EP   = dict(full=5, kd=6, lora=8, lorma=8, adalora=8, ia3=15, prompt=15)
LR   = dict(full=None, kd=None, lora=None, lorma=None, adalora=None, ia3=None, prompt=None)   
RANK = 16                       
COMMON = "--cw_pow 0.5 --window 1024 --max_sent 128 --bs 8"     
def X(m, extra=""):
    lr = f"--lr {LR[m]}" if LR[m] else ""
    return f"--method {m} {lr} {COMMON} {extra}"

## 6. Round A — LoRA (GPU0) ‖ LoRMA₊ (GPU1)

In [10]:
pA = launch(0, [("hyb_lora",  X("lora",  f"--rank {RANK} --alpha {2*RANK}"), EP["lora"], None)])
pB = launch(1, [("hyb_lorma", X("lorma", f"--rank {RANK} --alpha {2*RANK}"), EP["lorma"], None)])
wait([pA, pB]); display(table())

[hyb_lora] running |   [eval mid] step=156 dev macro-F1=0.2872 P=0.3515 R=0.2945
[hyb_lorma] running |   step 200/1248 ep 2 loss 1.7814 lr 9.82e-04 1.8min
[hyb_lora] running |   step 400/1248 ep 3 loss 1.1475 lr 8.42e-04 3.9min
[hyb_lorma] running |   step 400/1248 ep 3 loss 1.3790 lr 8.42e-04 3.6min
[hyb_lora] running |   step 600/1248 ep 4 loss 0.8005 lr 6.02e-04 5.8min
[hyb_lorma] running |   step 650/1248 ep 5 loss 0.9503 lr 5.34e-04 5.9min
[hyb_lora] running |   step 800/1248 ep 6 loss 0.4486 lr 3.32e-04 7.8min
[hyb_lorma] running |   step 850/1248 ep 6 loss 0.6287 lr 2.70e-04 7.6min
[hyb_lora] running |   step 1000/1248 ep 7 loss 0.2449 lr 1.12e-04 9.7min
[hyb_lorma] running |   [eval mid] step=1092 dev macro-F1=0.3911 P=0.4112 R=0.3885
[hyb_lora] running |   step 1200/1248 ep 8 loss 0.1492 lr 4.49e-06 11.6min
[hyb_lorma] done | == done in 11.6 min | dev macro-F1 = 0.4051
[hyb_lora] done | == done in 12.5 min | dev macro-F1 = 0.3910
[hyb_lorma] done | == done in 11.6 min | dev ma

,run,method,attn,dev_F1,P,R,CL,IT,trainable_M,train_min,steps,best_step,gpu_GB,infer_tok_s
1,hyb_lorma,lorma,causal,0.4051,0.4250,0.4005,0.3436,0.4201,3.231,11.1,1248/1248,1248,5.6,18310
0,hyb_lora,lora,causal,0.3910,0.4081,0.3885,0.3100,0.4202,2.776,12.0,1248/1248,1092,5.7,17748


## 7. Round B — AdaLoRA (GPU0) ‖ (IA)³ (GPU1)

In [11]:
pA = launch(0, [("hyb_adalora", X("adalora", "--ada_init_r 24 --ada_target_r 16 --alpha_ada 48"), EP["adalora"], None)])
pB = launch(1, [("hyb_ia3",     X("ia3"), EP["ia3"], None)])
wait([pA, pB]); display(table())

[hyb_adalora] running |   [eval mid] step=156 dev macro-F1=0.2822 P=0.3287 R=0.3065
[hyb_ia3] running |   step 200/2340 ep 2 loss 1.7539 lr 9.95e-03 1.7min
[hyb_adalora] running |   step 350/1248 ep 3 loss 1.5189 lr 8.88e-04 3.9min
[hyb_ia3] running |   step 450/2340 ep 3 loss 1.3251 lr 9.41e-03 3.7min
[hyb_adalora] running |   step 500/1248 ep 4 loss 1.2221 lr 7.30e-04 5.5min
[hyb_ia3] running |   step 700/2340 ep 5 loss 1.0691 lr 8.34e-03 5.9min
[hyb_adalora] running |   step 700/1248 ep 5 loss 0.9648 lr 4.66e-04 7.7min
[hyb_ia3] running |   step 900/2340 ep 6 loss 0.9278 lr 7.18e-03 7.5min
[hyb_adalora] running |   step 900/1248 ep 6 loss 0.7632 lr 2.11e-04 9.9min
[hyb_ia3] running |   step 1150/2340 ep 8 loss 0.8707 lr 5.50e-03 9.7min
[hyb_adalora] running |   step 1050/1248 ep 7 loss 0.6909 lr 7.23e-05 11.4min
[hyb_ia3] running |   step 1400/2340 ep 10 loss 0.7531 lr 3.76e-03 11.8min
[hyb_adalora] done | == done in 13.9 min | dev macro-F1 = 0.4105
[hyb_ia3] running |   step 1600/2

,run,method,attn,dev_F1,P,R,CL,IT,trainable_M,train_min,steps,best_step,gpu_GB,infer_tok_s
1,hyb_ia3,ia3,causal,0.4144,0.4106,0.4248,0.3614,0.4139,0.060,19.7,2340/2340,1872,5.3,19880
0,hyb_adalora,adalora,causal,0.4105,0.4115,0.4235,0.3428,0.4413,4.160,13.5,1248/1248,1248,5.8,17567
3,hyb_lorma,lorma,causal,0.4051,0.4250,0.4005,0.3436,0.4201,3.231,11.1,1248/1248,1248,5.6,18310
2,hyb_lora,lora,causal,0.3910,0.4081,0.3885,0.3100,0.4202,2.776,12.0,1248/1248,1092,5.7,17748


## 8. Round C — full fine-tuning = baseline + KD teacher (GPU0) ‖ Prompt tuning (GPU1)

In [12]:
pA = launch(0, [("hyb_full",   X("full"), EP["full"], None)])
pB = launch(1, [("hyb_prompt", X("prompt", "--n_prompt 32"), EP["prompt"], None)])
wait([pA, pB]); display(table())

[hyb_full] running |   step 200/780 ep 2 loss 1.5172 lr 4.74e-05 1.7min
[hyb_prompt] running |   step 250/2400 ep 2 loss 1.7616 lr 1.98e-02 1.7min
[hyb_full] running |   step 400/780 ep 3 loss 0.8188 lr 2.97e-05 3.6min
[hyb_prompt] running |   step 550/2400 ep 4 loss 1.4715 lr 1.82e-02 3.8min
[hyb_full] running |   step 650/780 ep 5 loss 0.3611 lr 4.44e-06 5.9min
[hyb_prompt] running |   step 850/2400 ep 6 loss 1.3507 lr 1.52e-02 5.9min
[hyb_full] done | == done in 7.4 min | dev macro-F1 = 0.3871
[hyb_prompt] running |   [eval mid] step=1120 dev macro-F1=0.3636 P=0.3798 R=0.3742
[hyb_full] done | == done in 7.4 min | dev macro-F1 = 0.3871
[hyb_prompt] running |   step 1400/2400 ep 9 loss 1.2951 lr 7.98e-03 9.7min
[hyb_full] done | == done in 7.4 min | dev macro-F1 = 0.3871
[hyb_prompt] running |   step 1700/2400 ep 11 loss 1.2241 lr 4.24e-03 11.8min
[hyb_full] done | == done in 7.4 min | dev macro-F1 = 0.3871
[hyb_prompt] running |   step 2000/2400 ep 13 loss 1.1432 lr 1.46e-03 13.9min

,run,method,attn,dev_F1,P,R,CL,IT,trainable_M,train_min,steps,best_step,gpu_GB,infer_tok_s
2,hyb_ia3,ia3,causal,0.4144,0.4106,0.4248,0.3614,0.4139,0.060,19.7,2340/2340,1872,5.3,19880
0,hyb_adalora,adalora,causal,0.4105,0.4115,0.4235,0.3428,0.4413,4.160,13.5,1248/1248,1248,5.8,17567
4,hyb_lorma,lorma,causal,0.4051,0.4250,0.4005,0.3436,0.4201,3.231,11.1,1248/1248,1248,5.6,18310
3,hyb_lora,lora,causal,0.3910,0.4081,0.3885,0.3100,0.4202,2.776,12.0,1248/1248,1092,5.7,17748
1,hyb_full,full,causal,0.3871,0.4154,0.3791,0.3019,0.4095,103.132,7.0,780/780,780,6.3,23551
5,hyb_prompt,prompt,causal,0.3743,0.3824,0.3846,0.3208,0.3775,0.036,16.6,2400/2400,2240,4.1,22960


## 9. Round D — KD (6-layer student, teacher = the `hyb_full` run above)

In [13]:
pA = launch(0, [("hyb_kd", X("kd", "--student_layers 6 --kd_T 2.0 --kd_alpha 0.5"), EP["kd"], "hyb_full")])
wait([pA]); display(table())

[hyb_kd] running |   step 250/936 ep 2 loss 1.6611 lr 9.24e-05 1.7min
[hyb_kd] running |   step 500/936 ep 4 loss 0.6761 lr 5.36e-05 3.6min
[hyb_kd] running |   step 800/936 ep 6 loss 0.2839 lr 6.48e-06 5.8min
[hyb_kd] done | == done in 7.0 min | dev macro-F1 = 0.4238
== hyb_kd: OK


,run,method,attn,dev_F1,P,R,CL,IT,trainable_M,train_min,steps,best_step,gpu_GB,infer_tok_s
3,hyb_kd,kd,causal,0.4238,0.4644,0.4117,0.3196,0.4465,56.469,6.8,936/936,936,4.0,45922
2,hyb_ia3,ia3,causal,0.4144,0.4106,0.4248,0.3614,0.4139,0.060,19.7,2340/2340,1872,5.3,19880
0,hyb_adalora,adalora,causal,0.4105,0.4115,0.4235,0.3428,0.4413,4.160,13.5,1248/1248,1248,5.8,17567
5,hyb_lorma,lorma,causal,0.4051,0.4250,0.4005,0.3436,0.4201,3.231,11.1,1248/1248,1248,5.6,18310
4,hyb_lora,lora,causal,0.3910,0.4081,0.3885,0.3100,0.4202,2.776,12.0,1248/1248,1092,5.7,17748
1,hyb_full,full,causal,0.3871,0.4154,0.3791,0.3019,0.4095,103.132,7.0,780/780,780,6.3,23551
6,hyb_prompt,prompt,causal,0.3743,0.3824,0.3846,0.3208,0.3775,0.036,16.6,2400/2400,2240,4.1,22960


## 10. Check against the official evaluator
`rr.py` computes macro-F1 itself (identical formula); this re-checks every run's dev predictions with the course's `evaluate_phase3.py` so you are sure the file format is what the scorer wants.

In [14]:
for d in sorted(glob.glob(f"{WORK}/runs/*/dev_pred.json")):
    n = d.split("/")[-2]
    r = subprocess.run(["python", "evaluate_phase3.py", "--truth-cl", f"{DATA}/truth_CL_dev.json", "--truth-it", f"{DATA}/truth_IT_dev.json", "--prediction", d, "--labels", f"{DATA}/label_vocab.json"], capture_output=True, text=True)
    print(f"{n:14s}", [l for l in r.stdout.splitlines() if "Macro-F1" in l] or r.stderr.strip())

hyb_adalora    ['Macro-F1:        0.410544']
hyb_full       ['Macro-F1:        0.387101']
hyb_ia3        ['Macro-F1:        0.414391']
hyb_kd         ['Macro-F1:        0.423825']
hyb_lora       ['Macro-F1:        0.391017']
hyb_lorma      ['Macro-F1:        0.405076']
hyb_prompt     ['Macro-F1:        0.374339']


## 11. Mix runs from another model, then ensembles
**Cross-model mixing.** Attach the *output* of the other notebook(s) (e.g. the decoder one) as a Kaggle dataset, point `OTHER` at its `runs` folder and run the cell: the log-prob files are copied in as `<tag>_<run>`, and `ensemble3.py --all` then averages across models too.
The runs are compatible because every notebook uses the same dev/test documents in the same order (the script checks ids and refuses to mix otherwise).
Single-method zips are already in `runs/<name>/submission.zip`. `--greedy` and `--bias_tune` are fitted **on dev**, so their dev score is optimistic – judge them by the leaderboard, and say so in the report.

In [15]:
OTHER = {}      
for tag, folder in OTHER.items():
    for d in sorted(glob.glob(f"{folder}/*/dev_lp.npz")):
        src = os.path.dirname(d); n = os.path.basename(src)
        if n.startswith("smoke"): continue
        dst = f"{WORK}/runs/" + (n if n.startswith(tag + "_") else f"{tag}_{n}"); os.makedirs(dst, exist_ok=True)      
        for f in ["dev_lp.npz", "dev_ids.json", "test_lp.npz", "test_ids.json", "results.json", "submission.zip"]:
            if os.path.exists(f"{src}/{f}"): shutil.copy(f"{src}/{f}", f"{dst}/{f}")
        print("imported", dst)

In [16]:
os.makedirs(f"{WORK}/submissions", exist_ok=True)
def ens(out, extra):
    r = subprocess.run(f"python ensemble3.py --runs_dir {WORK}/runs --data_dir {DATA} --out {WORK}/submissions/{out} {extra}", shell=True, capture_output=True, text=True)
    print(r.stdout[-2500:] + r.stderr[-800:])
    if os.path.exists(f"{WORK}/submissions/{out}/submission.zip"): shutil.copy(f"{WORK}/submissions/{out}/submission.zip", f"{WORK}/submissions/{out}.zip")
MINE = "hyb_lora,hyb_lorma,hyb_adalora,hyb_ia3,hyb_prompt"
ens("hyb_ens_peft", f"--names {MINE}")
ens("hyb_ens_all", f"--names {MINE},hyb_kd,hyb_full")


single runs (dev macro-F1):
  hyb_lora                     0.3910
  hyb_lorma                    0.4051
  hyb_adalora                  0.4105
  hyb_ia3                      0.4144
  hyb_prompt                   0.3743

ensemble of ['hyb_lora', 'hyb_lorma', 'hyb_adalora', 'hyb_ia3', 'hyb_prompt']
  dev macro-F1 = 0.4384
    Fact                     P=0.665 R=0.712 F1=0.688 support=444
    Issue                    P=0.538 R=0.467 F1=0.500 support=15
    ArgumentPetitioner       P=0.668 R=0.438 F1=0.529 support=400
    ArgumentRespondent       P=0.561 R=0.471 F1=0.512 support=272
    Statute                  P=0.733 R=0.800 F1=0.765 support=120
    Dissent                  P=0.000 R=0.000 F1=0.000 support=0
    PrecedentReliedUpon      P=0.493 R=0.421 F1=0.454 support=254
    PrecedentNotReliedUpon   P=0.299 R=0.460 F1=0.362 support=50
    PrecedentOverruled       P=0.000 R=0.000 F1=0.000 support=0
    RulingByLowerCourt       P=0.523 R=0.441 F1=0.479 support=102
    RatioOfTheDecision  

In [17]:
ens("hyb_ens_greedy", "--all --greedy")                 
ens("hyb_ens_greedy_bias", "--all --greedy --bias_tune")


single runs (dev macro-F1):
  hyb_adalora                  0.4105
  hyb_full                     0.3871
  hyb_ia3                      0.4144
  hyb_kd                       0.4238
  hyb_lora                     0.3910
  hyb_lorma                    0.4051
  hyb_prompt                   0.3743
  greedy step 1: +hyb_kd -> 0.4238
  greedy step 2: +hyb_adalora -> 0.4413

ensemble of ['hyb_kd', 'hyb_adalora']
  dev macro-F1 = 0.4413  (members chosen ON dev -> optimistic)
    Fact                     P=0.665 R=0.727 F1=0.695 support=444
    Issue                    P=0.667 R=0.400 F1=0.500 support=15
    ArgumentPetitioner       P=0.674 R=0.407 F1=0.508 support=400
    ArgumentRespondent       P=0.548 R=0.504 F1=0.525 support=272
    Statute                  P=0.767 R=0.767 F1=0.767 support=120
    Dissent                  P=0.000 R=0.000 F1=0.000 support=0
    PrecedentReliedUpon      P=0.469 R=0.531 F1=0.498 support=254
    PrecedentNotReliedUpon   P=0.314 R=0.440 F1=0.367 support=50
    

## 12. Collect the submission zips
Each zip contains only `prediction.json` at its root, with CL + IT documents merged (this is what Codabench expects). One zip per *method* = the distinct submissions the brief asks for; ensembles are extras.

In [18]:
for n in ['hyb_lora', 'hyb_lorma', 'hyb_adalora', 'hyb_ia3', 'hyb_prompt', 'hyb_kd', 'hyb_full']:
    s = f"{WORK}/runs/{n}/submission.zip"
    if os.path.exists(s): shutil.copy(s, f"{WORK}/submissions/{n}.zip")
import zipfile
for z in sorted(glob.glob(f"{WORK}/submissions/*.zip")):
    with zipfile.ZipFile(z) as f:
        p = json.load(f.open("prediction.json"))
    print(f"{os.path.basename(z):26s} files={zipfile.ZipFile(z).namelist()} docs={len(p)} sentences={sum(map(len, p.values()))}")

hyb_adalora.zip            files=['prediction.json'] docs=10 sentences=2197
hyb_ens_all.zip            files=['prediction.json'] docs=10 sentences=2197
hyb_ens_greedy.zip         files=['prediction.json'] docs=10 sentences=2197
hyb_ens_greedy_bias.zip    files=['prediction.json'] docs=10 sentences=2197
hyb_ens_peft.zip           files=['prediction.json'] docs=10 sentences=2197
hyb_full.zip               files=['prediction.json'] docs=10 sentences=2197
hyb_ia3.zip                files=['prediction.json'] docs=10 sentences=2197
hyb_kd.zip                 files=['prediction.json'] docs=10 sentences=2197
hyb_lora.zip               files=['prediction.json'] docs=10 sentences=2197
hyb_lorma.zip              files=['prediction.json'] docs=10 sentences=2197
hyb_prompt.zip             files=['prediction.json'] docs=10 sentences=2197


Download everything in `/kaggle/working/submissions/` and upload to Codabench (distinct submissions; ≤ 10 per day).
**Optional last step (only if time remains):** retrain the best config with `--full_data` (train+dev, no model selection → set epochs = the `best_step` epoch shown in the table). Its dev score is then in-sample and meaningless; compare on the leaderboard only.